# Vaani Boundary Detection — WavLM + BEATs (Verified Timestamps Only)

Clean, run-from-top notebook for noise-event boundary regression.

**Supervision:** only rows with `annotationQuality == "verified_timestamps"`; boundaries come only from `NoiseSubCategoryTimeStamp.start/end`.

**Model:** log-mel CNN + frozen WavLM Base+ + frozen BEATs Iter3+ AS2M → BiGRU → noise / onset-distance / offset-distance heads.

> Recommended: **Runtime → Restart session**, then run cells from top to bottom. Do not mix cells from older notebook versions.

## 0 — Setup

In [ ]:
!pip -q install -U librosa soundfile transformers pyarrow huggingface_hub

import os, sys, glob, json, ast, io, gc, random, time, zipfile, math, warnings, hashlib, shutil, subprocess, traceback
warnings.filterwarnings('ignore', message='datetime.datetime.utcnow.*')

import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import pyarrow.dataset as pads
import pyarrow.compute as pc
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy.ndimage import median_filter
from google.colab import drive

drive.mount('/content/drive')

# ---------------- paths ----------------
WORK = '/content/drive/MyDrive/Datathon/work'
CKPT_DIR = f'{WORK}/ckpt'
PRETRAINED_DIR = '/content/drive/MyDrive/Datathon/pretrained'
PQ_DIR = '/content/drive/MyDrive/Vaani/parquet'
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(PRETRAINED_DIR, exist_ok=True)

# ---------------- audio / training ----------------
SR = 16000
HOP = 160
NFFT = 512
N_MELS = 64
FRAME = HOP / SR
MAX_DIST = 0.5
SEED = 42
DEV_FRAC = 0.10

# RAM/VRAM-safe defaults for embedded parquet audio + WavLM + BEATs.
BATCH_SIZE = 1
NUM_WORKERS = 0
EPOCHS = 20
LR = 3e-4

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP_ENABLED = device.type == 'cuda'

print(f'device={device} | FRAME={FRAME:.3f}s | batch={BATCH_SIZE} | workers={NUM_WORKERS}')
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.6/294.6 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 796.8/796.8 kB 57.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.
Mounted at /content/drive
device=cuda | FRAME=0.010s | batch=1 | workers=0
GPU: NVIDIA L4


In [ ]:
# Optional memory monitor
import psutil

def show_memory(label=''):
    ram = psutil.Process(os.getpid()).memory_info().rss / 1024**3
    msg = f'{label} RAM={ram:.2f} GB'
    if torch.cuda.is_available():
        msg += (
            f' | GPU alloc={torch.cuda.memory_allocated()/1024**3:.2f} GB'
            f' | GPU reserved={torch.cuda.memory_reserved()/1024**3:.2f} GB'
        )
    print(msg)

show_memory('startup')

startup RAM=0.67 GB | GPU alloc=0.00 GB | GPU reserved=0.00 GB


## 1 — Load BEATs Iter3+ AS2M

In [ ]:
BEATS_REPO = '/content/unilm'
BEATS_DIR = f'{BEATS_REPO}/beats'
BEATS_CKPT = f'{PRETRAINED_DIR}/BEATs_iter3_plus_AS2M.pt'
BEATS_FILENAME = 'BEATs_iter3_plus_AS2M.pt'
BEATS_SHA256 = 'd43cbfad4d7b56381c061d7a24774f908d4d94c72961f6eb1d9090ff18cd8d34'

# Multiple mirrors; the first working mirror is cached permanently in Drive.
BEATS_HF_SOURCES = [
    ('lpepino/beats_ckpts', 'model'),
    ('mooneyko/BEATs', 'model'),
    ('nsivaku/nithin_checkpoints', 'model'),
    ('Bencr/beats-checkpoints', 'dataset'),
]

def sha256_file(path, chunk=8 * 1024 * 1024):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def valid_beats_file(path, verify_hash=True):
    if not os.path.isfile(path):
        return False
    if os.path.getsize(path) < 300 * 1024 * 1024:
        return False
    if verify_hash and sha256_file(path) != BEATS_SHA256:
        return False
    return True

def download_beats(dst):
    from huggingface_hub import hf_hub_download
    os.environ.setdefault('HF_HUB_DISABLE_XET', '1')
    errors=[]
    for repo_id, repo_type in BEATS_HF_SOURCES:
        print(f'Trying {repo_id} ...')
        try:
            cached = hf_hub_download(
                repo_id=repo_id,
                filename=BEATS_FILENAME,
                repo_type=repo_type,
                force_download=False,
            )
            if sha256_file(cached) != BEATS_SHA256:
                raise RuntimeError('checksum mismatch')
            tmp = dst + '.part'
            if os.path.exists(tmp): os.remove(tmp)
            shutil.copy2(cached, tmp)
            os.replace(tmp, dst)
            print('BEATs checkpoint saved:', dst)
            return
        except Exception as e:
            errors.append(f'{repo_id}: {type(e).__name__}: {e}')
            print(' failed:', errors[-1])
    raise RuntimeError('All BEATs mirrors failed:\n' + '\n'.join(errors))

# official Microsoft implementation
if not os.path.isfile(f'{BEATS_DIR}/BEATs.py'):
    if os.path.isdir(BEATS_REPO):
        shutil.rmtree(BEATS_REPO)
    subprocess.run(
        ['git','clone','--depth','1','https://github.com/microsoft/unilm.git',BEATS_REPO],
        check=True,
    )
if BEATS_DIR not in sys.path:
    sys.path.insert(0, BEATS_DIR)
from BEATs import BEATs, BEATsConfig

# checkpoint
stale = BEATS_CKPT + '.part'
if os.path.exists(stale): os.remove(stale)
if not valid_beats_file(BEATS_CKPT, verify_hash=True):
    if os.path.exists(BEATS_CKPT): os.remove(BEATS_CKPT)
    download_beats(BEATS_CKPT)
assert valid_beats_file(BEATS_CKPT, verify_hash=True), 'Invalid BEATs checkpoint.'
print('Using verified BEATs checkpoint:', BEATS_CKPT)

try:
    beats_ckpt = torch.load(BEATS_CKPT, map_location='cpu', weights_only=True)
except Exception:
    try:
        beats_ckpt = torch.load(BEATS_CKPT, map_location='cpu', weights_only=False)
    except TypeError:
        beats_ckpt = torch.load(BEATS_CKPT, map_location='cpu')

beats_cfg = BEATsConfig(beats_ckpt['cfg'])
BEATS_DIM = int(beats_cfg.encoder_embed_dim)
BEATS_MODEL = BEATs(beats_cfg)
BEATS_MODEL.load_state_dict(beats_ckpt['model'], strict=True)
BEATS_MODEL = BEATS_MODEL.to(device).eval()
for p in BEATS_MODEL.parameters():
    p.requires_grad_(False)

# release checkpoint dict before training
try:
    del beats_ckpt
except NameError:
    pass
gc.collect()

# frame-level smoke test
with torch.no_grad():
    _probe_wav = torch.randn(1, SR, device=device)
    _probe_pad = torch.zeros(1, SR, dtype=torch.bool, device=device)
    _probe_feat = BEATS_MODEL.extract_features(_probe_wav, padding_mask=_probe_pad)[0]

assert _probe_feat.ndim == 3 and _probe_feat.shape[-1] == BEATS_DIM
print('==============================')
print('BEATs READY')
print('==============================')
print('device       :', device)
print('feature shape:', tuple(_probe_feat.shape))
print('embedding dim:', BEATS_DIM)

# Compatibility aliases, so older variable names cannot break later cells.
AUDIOSET_ENCODER = BEATS_MODEL
AE_DIM = BEATS_DIM
AE_NAME = 'BEATs'
beats_model = BEATS_MODEL

del _probe_wav, _probe_pad, _probe_feat
if torch.cuda.is_available(): torch.cuda.empty_cache()
show_memory('after BEATs')

Using verified BEATs checkpoint: /content/drive/MyDrive/Datathon/pretrained/BEATs_iter3_plus_AS2M.pt


/usr/local/lib/python3.13/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


BEATs READY
device       : cuda
feature shape: (1, 48, 768)
embedding dim: 768
after BEATs RAM=1.43 GB | GPU alloc=0.36 GB | GPU reserved=0.41 GB


## 2 — Frozen WavLM / BEATs branches

In [ ]:
from transformers import WavLMModel

def make_valid_attention_mask(lengths, max_len, device, dtype=torch.long):
    idx = torch.arange(max_len, device=device).unsqueeze(0)
    return (idx < lengths.unsqueeze(1)).to(dtype)  # WavLM: 1 valid, 0 pad

def make_padding_mask(lengths, max_len, device):
    idx = torch.arange(max_len, device=device).unsqueeze(0)
    return idx >= lengths.unsqueeze(1)  # BEATs: True pad

class WavLMBranch(nn.Module):
    def __init__(self, name='microsoft/wavlm-base-plus'):
        super().__init__()
        self.m = WavLMModel.from_pretrained(name)
        for p in self.m.parameters(): p.requires_grad_(False)
        self.dim = int(self.m.config.hidden_size)
        self.m.eval()

    def forward(self, wav, T_target, wav_lengths=None):
        self.m.eval()
        attention_mask = None
        if wav_lengths is not None:
            attention_mask = make_valid_attention_mask(
                wav_lengths.to(wav.device), wav.shape[1], wav.device
            )
        # IMPORTANT: no_grad, not inference_mode; projection layers after this are trainable.
        with torch.no_grad():
            h = self.m(wav, attention_mask=attention_mask).last_hidden_state
        h = F.interpolate(
            h.transpose(1,2).float(), size=T_target, mode='linear', align_corners=False
        ).transpose(1,2)
        return h

class BEATsBranch(nn.Module):
    def __init__(self, model, dim):
        super().__init__()
        self.m = model
        self.dim = int(dim)
        self.m.eval()

    def forward(self, wav, T_target, wav_lengths=None):
        self.m.eval()
        padding_mask = None
        if wav_lengths is not None:
            padding_mask = make_padding_mask(
                wav_lengths.to(wav.device), wav.shape[1], wav.device
            )
        with torch.no_grad():
            h = self.m.extract_features(wav, padding_mask=padding_mask)[0]
        if h.ndim != 3:
            raise RuntimeError(f'BEATs returned {tuple(h.shape)}, expected [B,T,D].')
        h = F.interpolate(
            h.transpose(1,2).float(), size=T_target, mode='linear', align_corners=False
        ).transpose(1,2)
        return h

print('Frozen encoder branches defined | BEATs dim=', BEATS_DIM)

Frozen encoder branches defined | BEATs dim= 768


## 3 — Model

In [ ]:
class BoundaryModel(nn.Module):
    def __init__(self, n_mels=N_MELS, ch=(32,64,128,128), rnn=192,
                 wavlm_proj=128, beats_proj=128):
        super().__init__()
        blocks=[]
        cin=1
        for c in ch:
            blocks += [
                nn.Conv2d(cin,c,3,padding=1),
                nn.BatchNorm2d(c),
                nn.ReLU(),
                nn.AvgPool2d((2,1)),
                nn.Dropout2d(0.1),
            ]
            cin=c
        self.cnn = nn.Sequential(*blocks)
        self.mel_feat = cin * (n_mels // (2**len(ch)))

        self.wavlm = WavLMBranch()
        self.wavlm_proj = nn.Sequential(
            nn.Linear(self.wavlm.dim, wavlm_proj), nn.LayerNorm(wavlm_proj), nn.ReLU()
        )
        self.beats = BEATsBranch(BEATS_MODEL, BEATS_DIM)
        self.beats_proj = nn.Sequential(
            nn.Linear(BEATS_DIM, beats_proj), nn.LayerNorm(beats_proj), nn.ReLU()
        )

        feat_dim = self.mel_feat + wavlm_proj + beats_proj
        self.rnn = nn.GRU(
            feat_dim, rnn, num_layers=2, batch_first=True,
            bidirectional=True, dropout=0.2,
        )
        self.noise_head = nn.Linear(2*rnn, 1)
        self.onset_head = nn.Linear(2*rnn, 1)
        self.offset_head = nn.Linear(2*rnn, 1)

    def forward(self, x_mel, wav, wav_lengths=None):
        h_mel = self.cnn(x_mel)
        B,C,Fq,T = h_mel.shape
        h_mel = h_mel.permute(0,3,1,2).reshape(B,T,C*Fq)
        h_wavlm = self.wavlm_proj(self.wavlm(wav, T, wav_lengths))
        h_beats = self.beats_proj(self.beats(wav, T, wav_lengths))
        h = torch.cat([h_mel, h_wavlm, h_beats], dim=-1)
        h,_ = self.rnn(h)
        noise_logits = self.noise_head(h).squeeze(-1)
        onset_dist = torch.sigmoid(self.onset_head(h).squeeze(-1))
        offset_dist = torch.sigmoid(self.offset_head(h).squeeze(-1))
        return noise_logits, onset_dist, offset_dist

print('BoundaryModel defined: Mel + WavLM + BEATs -> BiGRU -> 3 heads')

BoundaryModel defined: Mel + WavLM + BEATs -> BiGRU -> 3 heads


## 4 — Memory-safe verified-timestamp parquet loading

In [ ]:
QUALITY_COL = 'annotationQuality'
TIMESTAMP_COL = 'NoiseSubCategoryTimeStamp'
AUDIO_COL = 'audio'

if not os.path.isdir(PQ_DIR):
    raise FileNotFoundError(PQ_DIR)

# IMPORTANT: filter in Arrow BEFORE converting to pandas.
# Do not load all 90k audio blobs into pandas.
arrow_ds = pads.dataset(PQ_DIR, format='parquet')
print('Available columns:', arrow_ds.schema.names)

required = [AUDIO_COL, 'duration', 'language', QUALITY_COL, TIMESTAMP_COL]
missing = [c for c in required if c not in arrow_ds.schema.names]
if missing:
    raise KeyError(f'Missing required parquet columns: {missing}')

verified_table = arrow_ds.to_table(
    columns=required,
    filter=(pc.field(QUALITY_COL) == 'verified_timestamps'),
)
print('Rows marked verified_timestamps:', verified_table.num_rows)

# Only 11k-ish verified rows are materialized, not all ~90k audio blobs.
df_verified = verified_table.to_pandas().reset_index(drop=True)
del verified_table, arrow_ds
gc.collect()
show_memory('after verified parquet load')


def parse_timestamp_list(raw):
    if raw is None:
        return []
    if isinstance(raw, np.ndarray): raw = raw.tolist()
    if isinstance(raw, tuple): raw = list(raw)
    if isinstance(raw, list): return raw
    try:
        if pd.isna(raw): return []
    except Exception:
        pass
    if isinstance(raw, str):
        s = raw.strip()
        if not s or s.lower() in {'nan','none','null'} or s == '[]':
            return []
        try:
            obj = json.loads(s)
        except Exception:
            try:
                obj = ast.literal_eval(s)
            except Exception:
                return []
        return obj if isinstance(obj, list) else []
    return []


def clean_verified_events(raw, duration_hint=None):
    duration=None
    try:
        duration=float(duration_hint)
        if not np.isfinite(duration) or duration <= 0: duration=None
    except Exception:
        duration=None

    out=[]
    for ev in parse_timestamp_list(raw):
        if not isinstance(ev, dict) or 'start' not in ev or 'end' not in ev:
            continue
        try:
            s=float(ev['start']); e=float(ev['end'])
        except (TypeError,ValueError):
            continue
        if not np.isfinite(s) or not np.isfinite(e):
            continue
        s=max(0.0,s)
        if duration is not None:
            s=min(s,duration); e=min(e,duration)
        if e <= s:
            continue
        out.append({
            'category': str(ev.get('category','')).strip(),
            'tag': str(ev.get('tag','')).strip(),
            'start': float(s),
            'end': float(e),
        })
    return out

# All rows were Arrow-filtered, but verify again.
q = df_verified[QUALITY_COL].astype(str).str.strip().str.lower()
assert q.eq('verified_timestamps').all()

df_verified['verified_events'] = [
    clean_verified_events(ts, dur)
    for ts, dur in zip(df_verified[TIMESTAMP_COL], df_verified['duration'])
]

before = len(df_verified)
df_verified = df_verified[df_verified['verified_events'].map(len).gt(0)].reset_index(drop=True)
print('Verified rows before valid-event filter:', before)
print('Usable verified boundary clips        :', len(df_verified))
print('Total verified events                 :', int(df_verified['verified_events'].map(len).sum()))
assert len(df_verified) >= 2

# deterministic split AFTER quality filtering
rng = np.random.RandomState(SEED)
idx = np.arange(len(df_verified)); rng.shuffle(idx)
n_dev = max(1, int(round(DEV_FRAC*len(idx))))
n_dev = min(n_dev, len(idx)-1)
dev_idx = idx[:n_dev]; train_idx = idx[n_dev:]

df_train = df_verified.iloc[train_idx].reset_index(drop=True)
df_dev = df_verified.iloc[dev_idx].reset_index(drop=True)

for name,d in [('train',df_train),('dev',df_dev)]:
    assert d[QUALITY_COL].astype(str).str.strip().str.lower().eq('verified_timestamps').all()
    assert d['verified_events'].map(len).gt(0).all()
    print(f'{name}: {len(d)} clips | {int(d["verified_events"].map(len).sum())} events')

print('Example verified event:', df_train.iloc[0]['verified_events'])
show_memory('after labels + split')

Available columns: ['audio', 'imageFileName', 'state', 'district', 'duration', 'language', 'annotationQuality', 'isTranscriptionAvailable', 'transcript', 'NoiseCategory', 'NoiseSubCategoryTimeStamp']
Rows marked verified_timestamps: 11111
after verified parquet load RAM=7.44 GB | GPU alloc=0.36 GB | GPU reserved=0.41 GB
Verified rows before valid-event filter: 11111
Usable verified boundary clips        : 11101
Total verified events                 : 19957
train: 9991 clips | 18008 events
dev: 1110 clips | 1949 events
Example verified event: [{'category': 'phone_signal_alarm', 'tag': '<phone sound>', 'start': 4.688, 'end': 5.893}, {'category': 'human_non_speech', 'tag': '[breathing]', 'start': 8.163, 'end': 8.531}]
after labels + split RAM=7.45 GB | GPU alloc=0.36 GB | GPU reserved=0.41 GB


## 5 — Audio decoding, features, targets

In [ ]:
def load_audio_value(audio_value, target_sr=SR):
    # pyarrow scalar -> Python value
    if hasattr(audio_value, 'as_py'):
        audio_value = audio_value.as_py()

    y=None; sr=None
    if isinstance(audio_value, dict):
        if audio_value.get('array') is not None:
            y=np.asarray(audio_value['array'], dtype=np.float32)
            sr=int(audio_value.get('sampling_rate') or target_sr)
        elif audio_value.get('bytes') is not None:
            b=audio_value['bytes']
            if isinstance(b, memoryview): b=b.tobytes()
            if isinstance(b, np.ndarray): b=b.tobytes()
            with io.BytesIO(b) as bio:
                y,sr=sf.read(bio, dtype='float32', always_2d=False)
        elif audio_value.get('path'):
            path=str(audio_value['path'])
            if not os.path.exists(path):
                raise FileNotFoundError(f'Parquet audio path not found: {path}')
            if path.endswith('.npy'):
                y=np.load(path).astype(np.float32); sr=target_sr
            else:
                y,sr=sf.read(path, dtype='float32', always_2d=False)
        else:
            raise ValueError(f'Unsupported audio dict keys: {list(audio_value.keys())}')
    elif isinstance(audio_value, (bytes,bytearray,memoryview)):
        if isinstance(audio_value,memoryview): audio_value=audio_value.tobytes()
        with io.BytesIO(audio_value) as bio:
            y,sr=sf.read(bio, dtype='float32', always_2d=False)
    elif isinstance(audio_value,np.ndarray):
        y=audio_value.astype(np.float32); sr=target_sr
    elif isinstance(audio_value,str):
        if not os.path.exists(audio_value): raise FileNotFoundError(audio_value)
        if audio_value.endswith('.npy'):
            y=np.load(audio_value).astype(np.float32); sr=target_sr
        else:
            y,sr=sf.read(audio_value, dtype='float32', always_2d=False)
    else:
        raise TypeError(f'Unsupported audio type: {type(audio_value)}')

    y=np.asarray(y,dtype=np.float32)
    if y.ndim==2:
        # soundfile convention is usually [samples, channels]
        y=y.mean(axis=1 if y.shape[0] >= y.shape[1] else 0)
    y=y.squeeze()
    if y.ndim != 1: raise ValueError(f'Expected mono waveform, got {y.shape}')
    if sr is None: sr=target_sr
    if int(sr) != int(target_sr):
        y=librosa.resample(y, orig_sr=int(sr), target_sr=int(target_sr)).astype(np.float32)
    if len(y)==0: raise ValueError('Decoded waveform is empty.')
    return np.nan_to_num(y,nan=0.0,posinf=0.0,neginf=0.0).astype(np.float32)

mel_fb = librosa.filters.mel(sr=SR, n_fft=NFFT, n_mels=N_MELS).astype(np.float32)

def logmel(y):
    S = np.abs(librosa.stft(y, n_fft=NFFT, hop_length=HOP, center=True))**2
    return np.log(mel_fb @ S + 1e-8).astype(np.float32)


def parse_events(raw, dur):
    # verified_events dicts -> simple (start,end) tuples
    out=[]
    if not isinstance(raw,(list,tuple,np.ndarray)): return out
    for ev in raw:
        if not isinstance(ev,dict): continue
        try: s=float(ev['start']); e=float(ev['end'])
        except (KeyError,TypeError,ValueError): continue
        s=max(0.0,min(s,dur)); e=max(s,min(e,dur))
        if e>s: out.append((s,e))
    return out


def make_targets(events, n_frames, frame_dur=FRAME, max_dist=MAX_DIST):
    noise=np.zeros(n_frames,np.float32)
    onset_dist=np.full(n_frames,max_dist,np.float32)
    offset_dist=np.full(n_frames,max_dist,np.float32)
    times=np.arange(n_frames,dtype=np.float32)*frame_dur
    n_valid=0
    for ev in events:
        if isinstance(ev,dict):
            try: s=float(ev['start']); e=float(ev['end'])
            except Exception: continue
        elif isinstance(ev,(list,tuple,np.ndarray)) and len(ev)>=2:
            try: s=float(ev[0]); e=float(ev[1])
            except Exception: continue
        else:
            continue
        if not np.isfinite(s) or not np.isfinite(e) or e<=s: continue
        a=max(0,min(int(np.floor(s/frame_dur)),n_frames-1))
        b=max(a+1,min(int(np.ceil(e/frame_dur)),n_frames))
        noise[a:b]=1.0
        onset_dist=np.minimum(onset_dist,np.abs(times-s))
        offset_dist=np.minimum(offset_dist,np.abs(times-e))
        n_valid += 1
    if n_valid==0: raise ValueError('No valid events passed to make_targets.')
    # heads use sigmoid, so normalize distance into [0,1]
    return (
        noise,
        np.minimum(onset_dist,max_dist).astype(np.float32)/max_dist,
        np.minimum(offset_dist,max_dist).astype(np.float32)/max_dist,
    )

# quick target test
_tn,_to,_tf=make_targets([(1.0,2.0),(3.0,3.5)],500)
assert _tn[100]==1.0 and _tn[50]==0.0
assert _to[100] < 0.01 and _tf[200] < 0.01
print('audio decoder + logmel + verified targets ready')

audio decoder + logmel + verified targets ready


## 6 — Dataset and loaders

In [ ]:
class GoldDataset(Dataset):
    def __init__(self, df, aug=False):
        self.df=df.reset_index(drop=True)
        self.aug=aug
        required=[AUDIO_COL,QUALITY_COL,'verified_events']
        missing=[c for c in required if c not in self.df.columns]
        if missing: raise KeyError(f'Dataset missing columns: {missing}')
        q=self.df[QUALITY_COL].astype(str).str.strip().str.lower()
        if not q.eq('verified_timestamps').all():
            raise ValueError('GoldDataset accepts verified_timestamps rows only.')
        if not self.df['verified_events'].map(len).gt(0).all():
            raise ValueError('Every GoldDataset row must have >=1 verified event.')

    def __len__(self): return len(self.df)

    def __getitem__(self,i):
        r=self.df.iloc[i]
        y=load_audio_value(r[AUDIO_COL], target_sr=SR)
        wav_len=int(len(y)); dur=wav_len/float(SR)
        events=parse_events(r['verified_events'],dur)
        if not events:
            raise RuntimeError(f'No verified event survived audio-duration clipping at row {i}.')

        if self.aug:
            gain=np.random.uniform(0.90,1.10)
            y=np.clip(y*gain,-1.0,1.0).astype(np.float32)

        mel=logmel(y)
        mel=(mel-mel.mean())/(mel.std()+1e-5)
        T=mel.shape[-1]
        noise,on_d,off_d=make_targets(events,T)

        # pad waveform to exact mel timeline; wav_len preserves true valid length
        need=T*HOP
        yy=np.pad(y,(0,max(0,need-wav_len)))[:need].astype(np.float32)
        return (
            torch.from_numpy(mel).unsqueeze(0),
            torch.from_numpy(yy),
            torch.from_numpy(noise),
            torch.from_numpy(on_d),
            torch.from_numpy(off_d),
            float(dur),
            int(wav_len),
        )


def collate_verified(batch):
    T_max=max(b[0].shape[-1] for b in batch)
    S_max=T_max*HOP
    def pad_frames(t,val=0.0):
        p=T_max-t.shape[-1]
        return F.pad(t,(0,p),value=val) if p>0 else t

    mels=torch.stack([pad_frames(b[0]) for b in batch])
    wavs=torch.stack([F.pad(b[1],(0,S_max-b[1].shape[0])) for b in batch])
    noise=torch.stack([pad_frames(b[2],0.0) for b in batch])
    on_d=torch.stack([pad_frames(b[3],1.0) for b in batch])
    off_d=torch.stack([pad_frames(b[4],1.0) for b in batch])
    durs=[b[5] for b in batch]
    wav_lens=torch.tensor([min(b[6],S_max) for b in batch],dtype=torch.long)
    masks=torch.stack([
        torch.cat([
            torch.ones(b[0].shape[-1],dtype=torch.float32),
            torch.zeros(T_max-b[0].shape[-1],dtype=torch.float32),
        ]) for b in batch
    ])
    return mels,wavs,noise,on_d,off_d,durs,masks,wav_lens


ds_train=GoldDataset(df_train,aug=True)
ds_dev=GoldDataset(df_dev,aug=False)

# Consistent canonical names
train_loader=DataLoader(
    ds_train,batch_size=BATCH_SIZE,shuffle=True,collate_fn=collate_verified,
    num_workers=NUM_WORKERS,pin_memory=(device.type=='cuda'),persistent_workers=False,
)
dev_loader=DataLoader(
    ds_dev,batch_size=BATCH_SIZE,shuffle=False,collate_fn=collate_verified,
    num_workers=NUM_WORKERS,pin_memory=(device.type=='cuda'),persistent_workers=False,
)

# Compatibility aliases for older cells; both refer to the same loaders.
dl_train=train_loader
dl_dev=dev_loader

print(f'Dataset sizes: train={len(ds_train)} | dev={len(ds_dev)}')
print(f'Batches/epoch: train={len(train_loader)} | dev={len(dev_loader)}')

# Single-item check
_s=ds_train[0]
print('Single sample:')
for name,x in zip(['mel','waveform','noise','onset','offset','duration','wav_len'],_s):
    if torch.is_tensor(x): print(f'  {name:10s} {tuple(x.shape)} {x.dtype}')
    else: print(f'  {name:10s} {x}')

# One-batch check
_b=next(iter(train_loader))
print('One batch:')
for name,x in zip(['mel','waveform','noise','onset','offset','duration','mask','wav_len'],_b):
    if torch.is_tensor(x): print(f'  {name:10s} {tuple(x.shape)} {x.dtype}')
    else: print(f'  {name:10s} {type(x).__name__}')
assert _b[2].sum().item() > 0, 'No positive verified noise frames in smoke batch.'
del _s,_b
gc.collect()
show_memory('after dataset smoke test')

Dataset sizes: train=9991 | dev=1110
Batches/epoch: train=9991 | dev=1110
Single sample:
  mel        (1, 64, 1326) torch.float32
  waveform   (212160,) torch.float32
  noise      (1326,) torch.float32
  onset      (1326,) torch.float32
  offset     (1326,) torch.float32
  duration   13.258625
  wav_len    212138
One batch:
  mel        (1, 1, 64, 528) torch.float32
  waveform   (1, 84480) torch.float32
  noise      (1, 528) torch.float32
  onset      (1, 528) torch.float32
  offset     (1, 528) torch.float32
  duration   list
  mask       (1, 528) torch.float32
  wav_len    (1,) torch.int64
after dataset smoke test RAM=7.52 GB | GPU alloc=0.36 GB | GPU reserved=0.41 GB


## 7 — End-to-end model smoke test

In [ ]:
model=BoundaryModel().to(device)
trainable_params=[p for p in model.parameters() if p.requires_grad]
n_trainable=sum(p.numel() for p in trainable_params)
n_total=sum(p.numel() for p in model.parameters())
print(f'trainable params: {n_trainable/1e6:.2f}M / total {n_total/1e6:.2f}M')

_smoke=next(iter(train_loader))
_sm_mel=_smoke[0][:1].to(device)
_sm_wav=_smoke[1][:1].to(device)
_sm_len=_smoke[7][:1].to(device)
_target_T=_smoke[2][:1].shape[1]

model.eval()
with torch.no_grad():
    with torch.amp.autocast(device_type=device.type, enabled=AMP_ENABLED):
        _out=model(_sm_mel,_sm_wav,_sm_len)

print('model outputs:', [tuple(x.shape) for x in _out])
assert all(x.shape == (1,_target_T) for x in _out), (
    f'Output/target mismatch: outputs={[tuple(x.shape) for x in _out]}, target T={_target_T}'
)
print('END-TO-END SMOKE TEST PASSED')

del _smoke,_sm_mel,_sm_wav,_sm_len,_out
if torch.cuda.is_available(): torch.cuda.empty_cache()
gc.collect()
show_memory('after model smoke test')

config.json:   0%|          | 0.00/2.23k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  378MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/248 [00:00<?, ?it/s]

trainable params: 2.21M / total 186.91M


/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


model outputs: [(1, 1466), (1, 1466), (1, 1466)]
END-TO-END SMOKE TEST PASSED
after model smoke test RAM=8.07 GB | GPU alloc=0.72 GB | GPU reserved=0.80 GB


## 8 — Training

In [ ]:
def masked_losses(noise_logits,onset_pred,offset_pred,noise_tgt,onset_tgt,offset_tgt,mask):
    T=min(noise_logits.shape[1],noise_tgt.shape[1],mask.shape[1])
    m=mask[:,:T].float()
    denom=m.sum().clamp_min(1.0)

    bce=F.binary_cross_entropy_with_logits(
        noise_logits[:,:T],noise_tgt[:,:T],reduction='none'
    )
    l_on=F.smooth_l1_loss(onset_pred[:,:T],onset_tgt[:,:T],reduction='none')
    l_off=F.smooth_l1_loss(offset_pred[:,:T],offset_tgt[:,:T],reduction='none')
    return (
        (bce*m).sum()/denom,
        (l_on*m).sum()/denom,
        (l_off*m).sum()/denom,
    )

def compact_state_dict(model):
    frozen=('wavlm.m.','beats.m.')
    return {k:v.detach().cpu() for k,v in model.state_dict().items() if not k.startswith(frozen)}

opt=torch.optim.AdamW(trainable_params,lr=LR,weight_decay=1e-4)
sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=EPOCHS)
BEST_PATH=f'{CKPT_DIR}/boundary_wavlm_beats_verified_best.pt'
best_val_loss=float('inf')

for epoch in range(EPOCHS):
    model.train()
    train_losses=[]

    for step,(mels,wavs,noise,on_d,off_d,durs,masks,wav_lens) in enumerate(train_loader):
        mels=mels.to(device,non_blocking=True)
        wavs=wavs.to(device,non_blocking=True)
        noise=noise.to(device,non_blocking=True)
        on_d=on_d.to(device,non_blocking=True)
        off_d=off_d.to(device,non_blocking=True)
        masks=masks.to(device,non_blocking=True)
        wav_lens=wav_lens.to(device,non_blocking=True)

        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type=device.type,enabled=AMP_ENABLED):
            p_n,p_o,p_f=model(mels,wavs,wav_lens)
            lb,lo,lf=masked_losses(p_n,p_o,p_f,noise,on_d,off_d,masks)
            loss=lb+lo+lf
        loss.backward()
        torch.nn.utils.clip_grad_norm_(trainable_params,5.0)
        opt.step()
        train_losses.append(float(loss.detach().cpu()))

        if step and step % 500 == 0:
            print(f'  epoch {epoch+1:02d} step {step}/{len(train_loader)} loss={np.mean(train_losses[-100:]):.4f}', flush=True)

    model.eval()
    val_losses=[]
    with torch.inference_mode():
        for mels,wavs,noise,on_d,off_d,durs,masks,wav_lens in dev_loader:
            mels=mels.to(device,non_blocking=True)
            wavs=wavs.to(device,non_blocking=True)
            noise=noise.to(device,non_blocking=True)
            on_d=on_d.to(device,non_blocking=True)
            off_d=off_d.to(device,non_blocking=True)
            masks=masks.to(device,non_blocking=True)
            wav_lens=wav_lens.to(device,non_blocking=True)
            with torch.amp.autocast(device_type=device.type,enabled=AMP_ENABLED):
                p_n,p_o,p_f=model(mels,wavs,wav_lens)
                lb,lo,lf=masked_losses(p_n,p_o,p_f,noise,on_d,off_d,masks)
                vl=lb+lo+lf
            val_losses.append(float(vl.detach().cpu()))

    sched.step()
    tl=float(np.mean(train_losses)); vl=float(np.mean(val_losses)); tag=''
    if vl < best_val_loss:
        best_val_loss=vl; tag=' *best*'
        torch.save({
            'model':compact_state_dict(model),
            'best_val_loss':best_val_loss,
            'epoch':epoch+1,
            'encoder':'WavLM-base-plus + BEATs-Iter3+-AS2M',
            'labels':'verified_timestamps / NoiseSubCategoryTimeStamp.start,end',
        },BEST_PATH)
    print(f'epoch {epoch+1:02d} | train {tl:.4f} | val {vl:.4f} | lr {sched.get_last_lr()[0]:.6f}{tag}')
    show_memory(f'epoch {epoch+1}')

# restore best compact trainable state
try:
    saved=torch.load(BEST_PATH,map_location='cpu',weights_only=False)
except TypeError:
    saved=torch.load(BEST_PATH,map_location='cpu')
missing,unexpected=model.load_state_dict(saved['model'],strict=False)
bad_missing=[k for k in missing if not (k.startswith('wavlm.m.') or k.startswith('beats.m.'))]
if bad_missing or unexpected:
    raise RuntimeError(f'Checkpoint mismatch: missing={bad_missing[:10]}, unexpected={unexpected[:10]}')
model.eval()
print(f'best val loss: {saved["best_val_loss"]:.4f} at epoch {saved["epoch"]}')
print('checkpoint:',BEST_PATH)

model.safetensors: reconstructing file:   0%|          |  0.00B /  378MB            

model.safetensors: downloading bytes:           |  0.00B            

  epoch 01 step 500/9991 loss=nan


KeyboardInterrupt: 

In [ ]:
# ============================================================
# STABLE TRAINING CELL
# WavLM + BEATs frozen | trainable fusion/GRU/heads in FP32
# ============================================================

import os
import math
import gc
import numpy as np
import torch
import torch.nn.functional as F


# ------------------------------------------------------------
# Resolve loader names safely
# ------------------------------------------------------------
train_loader = globals().get("train_loader", None)

if train_loader is None:
    train_loader = globals().get("dl_train", None)

dev_loader = globals().get("dev_loader", None)

if dev_loader is None:
    dev_loader = globals().get("dl_dev", None)

assert train_loader is not None, "train_loader / dl_train not found"
assert dev_loader is not None, "dev_loader / dl_dev not found"


# ------------------------------------------------------------
# Training settings
# ------------------------------------------------------------
LR = 1e-4
GRAD_CLIP = 1.0

# IMPORTANT: disable FP16/autocast for this recurrent model
USE_AMP = False

print("Training settings:")
print("  device      :", device)
print("  learning rate:", LR)
print("  grad clip   :", GRAD_CLIP)
print("  mixed precision:", USE_AMP, "(FP32 forced for stability)")


# ------------------------------------------------------------
# Fresh model
#
# DO NOT continue from the model that already trained with NaNs.
# ------------------------------------------------------------
if device.type == "cuda":
    torch.cuda.empty_cache()

gc.collect()

model = BoundaryModel().to(device)


# ------------------------------------------------------------
# Frozen encoders must remain frozen
# ------------------------------------------------------------
for p in model.wavlm.parameters():
    p.requires_grad_(False)

for p in model.beats.parameters():
    p.requires_grad_(False)


# ------------------------------------------------------------
# Trainable parameters
# ------------------------------------------------------------
trainable_params = [
    p for p in model.parameters()
    if p.requires_grad
]

n_trainable = sum(
    p.numel()
    for p in trainable_params
)

n_total = sum(
    p.numel()
    for p in model.parameters()
)

print(
    f"\ntrainable params: "
    f"{n_trainable/1e6:.2f}M / "
    f"total {n_total/1e6:.2f}M"
)


# ============================================================
# Utility: finite-value check
# ============================================================
def assert_finite(x, name):

    if not torch.isfinite(x).all():

        bad = (~torch.isfinite(x)).sum().item()

        try:
            xmin = torch.nan_to_num(x.detach()).min().item()
            xmax = torch.nan_to_num(x.detach()).max().item()
        except Exception:
            xmin = xmax = None

        raise FloatingPointError(
            f"{name} contains {bad} NaN/Inf values | "
            f"shape={tuple(x.shape)} | "
            f"min={xmin} max={xmax}"
        )


# ============================================================
# Stable masked losses
#
# Important:
# Select valid frames BEFORE computing losses.
# Do not calculate loss on padding then multiply by zero.
# ============================================================
def masked_losses(
    noise_logits,
    onset_pred,
    offset_pred,
    noise_tgt,
    onset_tgt,
    offset_tgt,
    mask,
):

    T = min(
        noise_logits.shape[1],
        onset_pred.shape[1],
        offset_pred.shape[1],
        noise_tgt.shape[1],
        onset_tgt.shape[1],
        offset_tgt.shape[1],
        mask.shape[1],
    )

    noise_logits = noise_logits[:, :T].float()
    onset_pred   = onset_pred[:, :T].float()
    offset_pred  = offset_pred[:, :T].float()

    noise_tgt  = noise_tgt[:, :T].float()
    onset_tgt  = onset_tgt[:, :T].float()
    offset_tgt = offset_tgt[:, :T].float()

    valid = mask[:, :T].bool()

    n_valid = int(valid.sum().item())

    if n_valid == 0:
        raise RuntimeError(
            "Batch contains zero valid frames."
        )


    # --------------------------------------------------------
    # Check before loss
    # --------------------------------------------------------
    assert_finite(noise_logits, "noise_logits")
    assert_finite(onset_pred,   "onset_pred")
    assert_finite(offset_pred,  "offset_pred")

    assert_finite(noise_tgt,  "noise_target")
    assert_finite(onset_tgt,  "onset_target")
    assert_finite(offset_tgt, "offset_target")


    # --------------------------------------------------------
    # Only valid frames enter the loss
    # --------------------------------------------------------
    nl = noise_logits[valid]
    nt = noise_tgt[valid]

    op = onset_pred[valid]
    ot = onset_tgt[valid]

    fp = offset_pred[valid]
    ft = offset_tgt[valid]


    loss_bce = F.binary_cross_entropy_with_logits(
        nl,
        nt,
    )

    loss_on = F.smooth_l1_loss(
        op,
        ot,
    )

    loss_off = F.smooth_l1_loss(
        fp,
        ft,
    )


    assert_finite(loss_bce, "loss_bce")
    assert_finite(loss_on,  "loss_onset")
    assert_finite(loss_off, "loss_offset")

    return (
        loss_bce,
        loss_on,
        loss_off,
    )


# ============================================================
# Keep frozen encoders in eval mode
# ============================================================
def set_training_mode():

    model.train()

    # Parent model.train() may propagate to children.
    # Force pretrained frozen encoders back to eval.
    model.wavlm.eval()
    model.beats.eval()

    if hasattr(model.wavlm, "m"):
        model.wavlm.m.eval()

    if hasattr(model.beats, "m"):
        model.beats.m.eval()


# ============================================================
# END-TO-END SMOKE TEST BEFORE TRAINING
# ============================================================

print("\nRunning finite forward/loss smoke test...")

smoke = next(iter(train_loader))

mels, wavs, noise, on_d, off_d, durs, masks, wav_lens = smoke

mels = mels[:1].to(
    device,
    dtype=torch.float32
)

wavs = wavs[:1].to(
    device,
    dtype=torch.float32
)

noise = noise[:1].to(
    device,
    dtype=torch.float32
)

on_d = on_d[:1].to(
    device,
    dtype=torch.float32
)

off_d = off_d[:1].to(
    device,
    dtype=torch.float32
)

masks = masks[:1].to(device)

wav_lens = wav_lens[:1].to(device)


assert_finite(mels, "smoke mel")
assert_finite(wavs, "smoke waveform")
assert_finite(noise, "smoke noise target")
assert_finite(on_d, "smoke onset target")
assert_finite(off_d, "smoke offset target")


model.eval()

with torch.no_grad():

    # NO autocast
    p_noise, p_onset, p_offset = model(
        mels,
        wavs,
        wav_lens,
    )


print(
    "model outputs:",
    [
        tuple(p_noise.shape),
        tuple(p_onset.shape),
        tuple(p_offset.shape),
    ],
)


assert_finite(
    p_noise,
    "smoke noise output"
)

assert_finite(
    p_onset,
    "smoke onset output"
)

assert_finite(
    p_offset,
    "smoke offset output"
)


lb, lo, lf = masked_losses(
    p_noise,
    p_onset,
    p_offset,
    noise,
    on_d,
    off_d,
    masks,
)

smoke_loss = lb + lo + lf

assert_finite(
    smoke_loss,
    "smoke total loss"
)

print(
    f"smoke loss = {smoke_loss.item():.6f} | "
    f"BCE={lb.item():.6f} | "
    f"on={lo.item():.6f} | "
    f"off={lf.item():.6f}"
)

print("\nEND-TO-END FINITE SMOKE TEST PASSED")


del (
    smoke,
    mels,
    wavs,
    noise,
    on_d,
    off_d,
    masks,
    wav_lens,
    p_noise,
    p_onset,
    p_offset,
    lb,
    lo,
    lf,
    smoke_loss,
)

gc.collect()

if device.type == "cuda":
    torch.cuda.empty_cache()


# ============================================================
# Optimizer / scheduler
# ============================================================

optimizer = torch.optim.AdamW(
    trainable_params,
    lr=LR,
    weight_decay=1e-4,
    eps=1e-8,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS,
)


# ============================================================
# Checkpoint
# ============================================================

BEST_PATH = os.path.join(
    CKPT,
    "boundary_wavlm_beats_verified_stable_best.pt"
)

best_val_loss = float("inf")


# ============================================================
# TRAINING
# ============================================================

for epoch in range(1, EPOCHS + 1):

    # --------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------
    set_training_mode()

    train_losses = []

    for step, batch in enumerate(
        train_loader,
        start=1,
    ):

        (
            mels,
            wavs,
            noise,
            on_d,
            off_d,
            durs,
            masks,
            wav_lens,
        ) = batch


        mels = mels.to(
            device,
            dtype=torch.float32,
            non_blocking=True,
        )

        wavs = wavs.to(
            device,
            dtype=torch.float32,
            non_blocking=True,
        )

        noise = noise.to(
            device,
            dtype=torch.float32,
            non_blocking=True,
        )

        on_d = on_d.to(
            device,
            dtype=torch.float32,
            non_blocking=True,
        )

        off_d = off_d.to(
            device,
            dtype=torch.float32,
            non_blocking=True,
        )

        masks = masks.to(
            device,
            non_blocking=True,
        )

        wav_lens = wav_lens.to(
            device,
            non_blocking=True,
        )


        # ----------------------------------------------------
        # Input checks
        # ----------------------------------------------------
        assert_finite(
            mels,
            f"epoch {epoch} step {step} mel"
        )

        assert_finite(
            wavs,
            f"epoch {epoch} step {step} wav"
        )

        assert_finite(
            noise,
            f"epoch {epoch} step {step} noise tgt"
        )

        assert_finite(
            on_d,
            f"epoch {epoch} step {step} onset tgt"
        )

        assert_finite(
            off_d,
            f"epoch {epoch} step {step} offset tgt"
        )


        optimizer.zero_grad(
            set_to_none=True
        )


        # ====================================================
        # FP32 FORWARD — NO AUTOCAST
        # ====================================================
        p_noise, p_onset, p_offset = model(
            mels,
            wavs,
            wav_lens,
        )


        assert_finite(
            p_noise,
            f"epoch {epoch} step {step} noise logits"
        )

        assert_finite(
            p_onset,
            f"epoch {epoch} step {step} onset output"
        )

        assert_finite(
            p_offset,
            f"epoch {epoch} step {step} offset output"
        )


        loss_bce, loss_on, loss_off = masked_losses(
            p_noise,
            p_onset,
            p_offset,
            noise,
            on_d,
            off_d,
            masks,
        )

        loss = (
            loss_bce
            + loss_on
            + loss_off
        )


        # ----------------------------------------------------
        # NEVER backpropagate NaN
        # ----------------------------------------------------
        if not torch.isfinite(loss):

            raise FloatingPointError(
                f"\nNON-FINITE LOSS BEFORE BACKWARD\n"
                f"epoch={epoch} step={step}\n"
                f"BCE={loss_bce.item()}\n"
                f"onset={loss_on.item()}\n"
                f"offset={loss_off.item()}"
            )


        # ----------------------------------------------------
        # Backward
        # ----------------------------------------------------
        loss.backward()


        # ----------------------------------------------------
        # Gradient clipping
        # ----------------------------------------------------
        grad_norm = torch.nn.utils.clip_grad_norm_(
            trainable_params,
            GRAD_CLIP,
            error_if_nonfinite=True,
        )


        if not torch.isfinite(grad_norm):

            raise FloatingPointError(
                f"Non-finite gradient norm "
                f"at epoch={epoch}, step={step}"
            )


        optimizer.step()


        train_losses.append(
            loss.detach().item()
        )


        # ----------------------------------------------------
        # Logging
        # ----------------------------------------------------
        if step <= 5 or step % 500 == 0:

            recent = np.mean(
                train_losses[-min(
                    100,
                    len(train_losses)
                ):]
            )

            print(
                f"epoch {epoch:02d} "
                f"step {step}/{len(train_loader)} "
                f"loss={recent:.5f} "
                f"(bce={loss_bce.item():.4f}, "
                f"on={loss_on.item():.4f}, "
                f"off={loss_off.item():.4f}, "
                f"g={float(grad_norm):.3f})",
                flush=True,
            )


    # --------------------------------------------------------
    # Scheduler
    # --------------------------------------------------------
    scheduler.step()


    # ========================================================
    # VALIDATION
    # ========================================================

    model.eval()

    val_losses = []

    with torch.no_grad():

        for batch in dev_loader:

            (
                mels,
                wavs,
                noise,
                on_d,
                off_d,
                durs,
                masks,
                wav_lens,
            ) = batch


            mels = mels.to(
                device,
                dtype=torch.float32,
                non_blocking=True,
            )

            wavs = wavs.to(
                device,
                dtype=torch.float32,
                non_blocking=True,
            )

            noise = noise.to(
                device,
                dtype=torch.float32,
                non_blocking=True,
            )

            on_d = on_d.to(
                device,
                dtype=torch.float32,
                non_blocking=True,
            )

            off_d = off_d.to(
                device,
                dtype=torch.float32,
                non_blocking=True,
            )

            masks = masks.to(
                device,
                non_blocking=True,
            )

            wav_lens = wav_lens.to(
                device,
                non_blocking=True,
            )


            p_noise, p_onset, p_offset = model(
                mels,
                wavs,
                wav_lens,
            )


            lb, lo, lf = masked_losses(
                p_noise,
                p_onset,
                p_offset,
                noise,
                on_d,
                off_d,
                masks,
            )


            val_loss = lb + lo + lf

            if not torch.isfinite(val_loss):

                raise FloatingPointError(
                    "Non-finite validation loss."
                )


            val_losses.append(
                val_loss.item()
            )


    train_loss = float(
        np.mean(train_losses)
    )

    val_loss = float(
        np.mean(val_losses)
    )


    # ========================================================
    # Save best
    # ========================================================

    tag = ""

    if val_loss < best_val_loss:

        best_val_loss = val_loss
        tag = " *BEST*"


        state = {
            k: v.detach().cpu()
            for k, v in model.state_dict().items()
            if (
                not k.startswith("wavlm.")
                and not k.startswith("beats.")
            )
        }


        torch.save(
            {
                "model": state,
                "epoch": epoch,
                "best_val_loss": best_val_loss,
                "lr": LR,
                "encoder":
                    "WavLM-base-plus + BEATs-Iter3+-AS2M",
                "annotation":
                    "verified_timestamps",
            },
            BEST_PATH,
        )


    print(
        f"\nepoch {epoch:02d} | "
        f"train {train_loss:.6f} | "
        f"val {val_loss:.6f} | "
        f"lr {scheduler.get_last_lr()[0]:.7f}"
        f"{tag}"
    )


    # --------------------------------------------------------
    # Memory report
    # --------------------------------------------------------
    if device.type == "cuda":

        print(
            f"GPU allocated="
            f"{torch.cuda.memory_allocated()/1024**3:.2f} GB | "
            f"reserved="
            f"{torch.cuda.memory_reserved()/1024**3:.2f} GB"
        )

        torch.cuda.empty_cache()

    gc.collect()


print("\nTraining complete.")
print("Best validation loss:", best_val_loss)
print("Best checkpoint:", BEST_PATH)

In [ ]:
# ============================================================
# Checkpoint directory
# ============================================================

import os

WORK = "/content/drive/MyDrive/Datathon/work"
CKPT = os.path.join(WORK, "ckpt")

os.makedirs(CKPT, exist_ok=True)

print("Checkpoint directory:", CKPT)

Checkpoint directory: /content/drive/MyDrive/Datathon/work/ckpt


In [ ]:
BEST_PATH = os.path.join(
    CKPT,
    "boundary_wavlm_beats_verified_stable_best.pt"
)

print("Best checkpoint will be saved to:")
print(BEST_PATH)

Best checkpoint will be saved to:
/content/drive/MyDrive/Datathon/work/ckpt/boundary_wavlm_beats_verified_stable_best.pt


In [ ]:
# ============================================================
# STABLE TRAINING CELL
# ============================================================

import os
import gc
import numpy as np
import torch
import torch.nn.functional as F

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
WORK = "/content/drive/MyDrive/Datathon/work"
CKPT = os.path.join(WORK, "ckpt")
os.makedirs(CKPT, exist_ok=True)

BEST_PATH = os.path.join(
    CKPT,
    "boundary_wavlm_beats_verified_stable_best.pt"
)

print("Checkpoint:", BEST_PATH)

# ------------------------------------------------------------
# Loader aliases
# ------------------------------------------------------------
train_loader = globals().get("train_loader", globals().get("dl_train", None))
dev_loader   = globals().get("dev_loader", globals().get("dl_dev", None))

assert train_loader is not None, "train_loader / dl_train missing"
assert dev_loader is not None, "dev_loader / dl_dev missing"

# ------------------------------------------------------------
# Hyperparameters
# ------------------------------------------------------------
LR = 1e-4
GRAD_CLIP = 1.0

if "EPOCHS" not in globals():
    EPOCHS = 20

print("LR:", LR)
print("EPOCHS:", EPOCHS)
print("FP32 training enabled")

# ------------------------------------------------------------
# Fresh model
# ------------------------------------------------------------
if "model" not in globals():
    model = BoundaryModel().to(device)

trainable_params = [
    p for p in model.parameters()
    if p.requires_grad
]

print(
    f"trainable params: "
    f"{sum(p.numel() for p in trainable_params)/1e6:.2f}M"
)

# ------------------------------------------------------------
# Finite checker
# ------------------------------------------------------------
def check_finite(x, name):
    if not torch.isfinite(x).all():
        raise FloatingPointError(
            f"{name} contains NaN/Inf"
        )

# ------------------------------------------------------------
# Stable masked loss
# ------------------------------------------------------------
def masked_losses(
    noise_logits,
    onset_pred,
    offset_pred,
    noise_tgt,
    onset_tgt,
    offset_tgt,
    mask
):

    T = min(
        noise_logits.shape[1],
        onset_pred.shape[1],
        offset_pred.shape[1],
        noise_tgt.shape[1],
        onset_tgt.shape[1],
        offset_tgt.shape[1],
        mask.shape[1]
    )

    noise_logits = noise_logits[:, :T].float()
    onset_pred   = onset_pred[:, :T].float()
    offset_pred  = offset_pred[:, :T].float()

    noise_tgt  = noise_tgt[:, :T].float()
    onset_tgt  = onset_tgt[:, :T].float()
    offset_tgt = offset_tgt[:, :T].float()

    valid = mask[:, :T].bool()

    if valid.sum() == 0:
        raise RuntimeError("No valid frames in batch")

    check_finite(noise_logits, "noise_logits")
    check_finite(onset_pred, "onset_pred")
    check_finite(offset_pred, "offset_pred")

    bce = F.binary_cross_entropy_with_logits(
        noise_logits[valid],
        noise_tgt[valid]
    )

    l_on = F.smooth_l1_loss(
        onset_pred[valid],
        onset_tgt[valid]
    )

    l_off = F.smooth_l1_loss(
        offset_pred[valid],
        offset_tgt[valid]
    )

    return bce, l_on, l_off

# ------------------------------------------------------------
# Keep frozen encoders in eval mode
# ------------------------------------------------------------
def set_train_mode():
    model.train()

    if hasattr(model, "wavlm"):
        model.wavlm.eval()
        if hasattr(model.wavlm, "m"):
            model.wavlm.m.eval()

    if hasattr(model, "beats"):
        model.beats.eval()
        if hasattr(model.beats, "m"):
            model.beats.m.eval()

# ------------------------------------------------------------
# Optimizer
# ------------------------------------------------------------
optimizer = torch.optim.AdamW(
    trainable_params,
    lr=LR,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS
)

best_val_loss = float("inf")

# ============================================================
# TRAIN
# ============================================================

for epoch in range(1, EPOCHS + 1):

    set_train_mode()

    train_losses = []

    for step, batch in enumerate(
        train_loader,
        start=1
    ):

        (
            mels,
            wavs,
            noise,
            on_d,
            off_d,
            durs,
            masks,
            wav_lens
        ) = batch

        mels = mels.to(
            device,
            dtype=torch.float32,
            non_blocking=True
        )

        wavs = wavs.to(
            device,
            dtype=torch.float32,
            non_blocking=True
        )

        noise = noise.to(
            device,
            dtype=torch.float32,
            non_blocking=True
        )

        on_d = on_d.to(
            device,
            dtype=torch.float32,
            non_blocking=True
        )

        off_d = off_d.to(
            device,
            dtype=torch.float32,
            non_blocking=True
        )

        masks = masks.to(
            device,
            non_blocking=True
        )

        wav_lens = wav_lens.to(
            device,
            non_blocking=True
        )

        check_finite(mels, "mel")
        check_finite(wavs, "wav")
        check_finite(noise, "noise target")
        check_finite(on_d, "onset target")
        check_finite(off_d, "offset target")

        optimizer.zero_grad(set_to_none=True)

        # FP32 forward, NO autocast
        p_noise, p_onset, p_offset = model(
            mels,
            wavs,
            wav_lens
        )

        check_finite(p_noise, "noise output")
        check_finite(p_onset, "onset output")
        check_finite(p_offset, "offset output")

        loss_bce, loss_on, loss_off = masked_losses(
            p_noise,
            p_onset,
            p_offset,
            noise,
            on_d,
            off_d,
            masks
        )

        loss = (
            loss_bce
            + loss_on
            + loss_off
        )

        if not torch.isfinite(loss):
            raise FloatingPointError(
                f"NaN loss at epoch {epoch}, step {step}"
            )

        loss.backward()

        grad_norm = torch.nn.utils.clip_grad_norm_(
            trainable_params,
            GRAD_CLIP,
            error_if_nonfinite=True
        )

        optimizer.step()

        train_losses.append(
            loss.item()
        )

        if step <= 5 or step % 500 == 0:

            recent = np.mean(
                train_losses[-100:]
            )

            print(
                f"epoch {epoch:02d} "
                f"step {step}/{len(train_loader)} "
                f"loss={recent:.5f} "
                f"| bce={loss_bce.item():.4f} "
                f"on={loss_on.item():.4f} "
                f"off={loss_off.item():.4f} "
                f"g={float(grad_norm):.3f}",
                flush=True
            )

    scheduler.step()

    # ========================================================
    # VALIDATION
    # ========================================================

    model.eval()

    val_losses = []

    with torch.no_grad():

        for batch in dev_loader:

            (
                mels,
                wavs,
                noise,
                on_d,
                off_d,
                durs,
                masks,
                wav_lens
            ) = batch

            mels = mels.to(
                device,
                dtype=torch.float32
            )

            wavs = wavs.to(
                device,
                dtype=torch.float32
            )

            noise = noise.to(
                device,
                dtype=torch.float32
            )

            on_d = on_d.to(
                device,
                dtype=torch.float32
            )

            off_d = off_d.to(
                device,
                dtype=torch.float32
            )

            masks = masks.to(device)
            wav_lens = wav_lens.to(device)

            p_noise, p_onset, p_offset = model(
                mels,
                wavs,
                wav_lens
            )

            lb, lo, lf = masked_losses(
                p_noise,
                p_onset,
                p_offset,
                noise,
                on_d,
                off_d,
                masks
            )

            val_loss = lb + lo + lf

            if not torch.isfinite(val_loss):
                raise FloatingPointError(
                    "Validation loss became NaN/Inf"
                )

            val_losses.append(
                val_loss.item()
            )

    train_loss = float(
        np.mean(train_losses)
    )

    val_loss = float(
        np.mean(val_losses)
    )

    tag = ""

    # --------------------------------------------------------
    # Save best model
    # --------------------------------------------------------
    if val_loss < best_val_loss:

        best_val_loss = val_loss
        tag = " *BEST*"

        state = {
            k: v.detach().cpu()
            for k, v in model.state_dict().items()
            if (
                not k.startswith("wavlm.")
                and not k.startswith("beats.")
            )
        }

        torch.save(
            {
                "model": state,
                "epoch": epoch,
                "best_val_loss": best_val_loss,
                "encoder":
                    "WavLM-base-plus + BEATs-Iter3+-AS2M",
                "annotation":
                    "verified_timestamps"
            },
            BEST_PATH
        )

    print(
        f"\nepoch {epoch:02d} "
        f"| train {train_loss:.6f} "
        f"| val {val_loss:.6f} "
        f"| lr {scheduler.get_last_lr()[0]:.7f}"
        f"{tag}"
    )

    if device.type == "cuda":
        print(
            f"GPU alloc="
            f"{torch.cuda.memory_allocated()/1024**3:.2f} GB "
            f"| reserved="
            f"{torch.cuda.memory_reserved()/1024**3:.2f} GB"
        )
        torch.cuda.empty_cache()

    gc.collect()

print("\nTraining complete")
print("Best validation loss:", best_val_loss)
print("Checkpoint:", BEST_PATH)

Checkpoint: /content/drive/MyDrive/Datathon/work/ckpt/boundary_wavlm_beats_verified_stable_best.pt
LR: 0.0001
EPOCHS: 20
FP32 training enabled
trainable params: 2.21M
epoch 01 step 1/9991 loss=0.89156 | bce=0.6699 on=0.0944 off=0.1273 g=4.217
epoch 01 step 2/9991 loss=0.85087 | bce=0.6142 on=0.0932 off=0.1028 g=1.899
epoch 01 step 3/9991 loss=0.77495 | bce=0.4329 on=0.0880 off=0.1021 g=3.093
epoch 01 step 4/9991 loss=0.71348 | bce=0.3581 on=0.0751 off=0.0958 g=2.507
epoch 01 step 5/9991 loss=0.68571 | bce=0.4152 on=0.0772 off=0.0822 g=1.656
epoch 01 step 500/9991 loss=0.64355 | bce=0.4401 on=0.0443 off=0.0252 g=1.395
epoch 01 step 1000/9991 loss=0.54512 | bce=0.0988 on=0.0139 off=0.0189 g=0.504
epoch 01 step 1500/9991 loss=0.48786 | bce=0.4721 on=0.0025 off=0.0567 g=3.030
epoch 01 step 2000/9991 loss=0.47060 | bce=0.6542 on=0.0229 off=0.0187 g=12.222
epoch 01 step 2500/9991 loss=0.56247 | bce=0.0426 on=0.0021 off=0.0015 g=0.290
epoch 01 step 3000/9991 loss=0.41795 | bce=0.1149 on=0.007

## 9 — Decode and boundary snapping

In [ ]:
def decode_standard(p,dur,thr=0.85,med_ms=200,min_dur=0.12,merge_gap=0.10):
    m=p>=thr
    k=max(1,int(round((med_ms/1000)/FRAME)))
    if k>1:
        if k%2==0: k+=1
        m=median_filter(m.astype(np.float32),size=k)>0.5
    ev=[]; s=None
    for i,v in enumerate(m):
        if v and s is None: s=i
        elif (not v) and s is not None:
            ev.append([s*FRAME,i*FRAME]); s=None
    if s is not None: ev.append([s*FRAME,len(m)*FRAME])
    merged=[]
    for e in ev:
        if merged and e[0]-merged[-1][1] <= merge_gap: merged[-1][1]=e[1]
        else: merged.append(e)
    return [(a,min(b,dur)) for a,b in merged if min(b,dur)-a>=min_dur]

def snap_boundaries(candidates,onset_dist,offset_dist,dur,snap_window=0.20,min_dur=0.12):
    out=[]; W=max(1,int(round(snap_window/FRAME))); N=len(onset_dist)
    for on,off in candidates:
        on_f=int(round(on/FRAME)); off_f=int(round(off/FRAME))
        a=max(0,on_f-W); b=min(N,on_f+W+1)
        best_on=(a+int(np.argmin(onset_dist[a:b])))*FRAME if b>a else on
        a=max(0,off_f-W); b=min(N,off_f+W+1)
        best_off=(a+int(np.argmin(offset_dist[a:b])))*FRAME if b>a else off
        best_on=max(0.0,best_on); best_off=min(float(dur),best_off)
        if best_off-best_on>=min_dur: out.append((best_on,best_off))
    return out

print('decoder ready')

decoder ready


## 10 — Dev F1: snap vs no-snap

In [ ]:
def event_f1(pred,ref,tol=0.2):
    tp=0; matched=set()
    for po,pf in pred:
        for j,(ro,rf) in enumerate(ref):
            if j in matched: continue
            rd=max(rf-ro,1e-6); t=tol*rd
            if abs(po-ro)<=t and abs(pf-rf)<=t:
                tp+=1; matched.add(j); break
    pr=tp/max(len(pred),1); rc=tp/max(len(ref),1)
    return 2*pr*rc/max(pr+rc,1e-9)

P_NAT={'thr':0.85,'med_ms':200,'min_dur':0.12,'merge_gap':0.10}
f1_ns=[]; f1_sn=[]
model.eval()
with torch.inference_mode():
    for mels,wavs,noise,on_d,off_d,durs,masks,wav_lens in dev_loader:
        mels=mels.to(device,non_blocking=True)
        wavs=wavs.to(device,non_blocking=True)
        wav_lens=wav_lens.to(device,non_blocking=True)
        with torch.amp.autocast(device_type=device.type,enabled=AMP_ENABLED):
            p_n,p_o,p_f=model(mels,wavs,wav_lens)
        for b in range(mels.shape[0]):
            T=int(masks[b].sum().item()); dur=float(durs[b])
            pn=torch.sigmoid(p_n[b,:T]).float().cpu().numpy()
            po=p_o[b,:T].float().cpu().numpy()
            pf=p_f[b,:T].float().cpu().numpy()
            gt=noise[b,:T].numpy()

            gt_ev=[]; in_ev=False; s_t=0
            for t,v in enumerate(gt):
                if v>0.5 and not in_ev: s_t=t; in_ev=True
                elif v<=0.5 and in_ev: gt_ev.append((s_t*FRAME,t*FRAME)); in_ev=False
            if in_ev: gt_ev.append((s_t*FRAME,len(gt)*FRAME))

            cands=decode_standard(pn,dur,**P_NAT)
            f1_ns.append(event_f1(cands,gt_ev))
            f1_sn.append(event_f1(snap_boundaries(cands,po,pf,dur),gt_ev))

f1_no=float(np.mean(f1_ns)); f1_yes=float(np.mean(f1_sn))
print(f'dev F1 WITHOUT snap: {f1_no:.4f}')
print(f'dev F1 WITH    snap: {f1_yes:.4f}')
print(f'delta:               {f1_yes-f1_no:+.4f}')
SNAP = f1_yes >= f1_no - 0.005
print('use snapping:',SNAP)

dev F1 WITHOUT snap: 0.0000
dev F1 WITH    snap: 0.0000
delta:               +0.0000
use snapping: True


In [ ]:
TEST_ROOT = '/content/drive/MyDrive/PHD/Datathon/'

## 11 — Test inference

In [ ]:
TEST_ROOT='/content/drive/MyDrive/PHD/Datathon/'
wavs_test=sorted(glob.glob(f'{TEST_ROOT}/*.wav',recursive=False))
if not wavs_test:
    input_zip='/content/drive/MyDrive/Datathon/input_data.zip'
    if not os.path.isfile(input_zip):
        raise FileNotFoundError(f'No test WAVs under {TEST_ROOT} and no {input_zip}')
    os.makedirs(TEST_ROOT,exist_ok=True)
    with zipfile.ZipFile(input_zip) as z: z.extractall(TEST_ROOT)
    wavs_test=sorted(glob.glob(f'{TEST_ROOT}/**/*.wav',recursive=True))

cid_of=lambda p: os.path.splitext(os.path.basename(p))[0]
paths={cid_of(p):p for p in wavs_test}
print(len(wavs_test),'test clips')

RUN_TAG=time.strftime('%Y%m%dT%H%M%S')
PRED={}; model.eval(); t0=time.time()
with torch.inference_mode():
    for i,(cid,path) in enumerate(sorted(paths.items())):
        y,sr0=sf.read(path,dtype='float32')
        if y.ndim>1: y=y.mean(axis=1)
        if sr0!=SR: y=librosa.resample(y,orig_sr=sr0,target_sr=SR).astype(np.float32)
        valid_samples=len(y); dur=valid_samples/SR
        mel=logmel(y); mel=(mel-mel.mean())/(mel.std()+1e-5); T=mel.shape[1]
        need=T*HOP
        yy=np.pad(y,(0,max(0,need-valid_samples)))[:need].astype(np.float32)
        xt=torch.from_numpy(mel)[None,None].to(device)
        wt=torch.from_numpy(yy)[None].to(device)
        wl=torch.tensor([min(valid_samples,need)],dtype=torch.long,device=device)
        with torch.amp.autocast(device_type=device.type,enabled=AMP_ENABLED):
            p_n,p_o,p_f=model(xt,wt,wl)
        pn=torch.sigmoid(p_n[0]).float().cpu().numpy()
        po=p_o[0].float().cpu().numpy(); pf=p_f[0].float().cpu().numpy()
        cands=decode_standard(pn,dur,**P_NAT)
        PRED[cid]=snap_boundaries(cands,po,pf,dur) if SNAP else cands
        if i%500==0: print(f'  {i}/{len(paths)} | {(time.time()-t0)/60:.1f} min',flush=True)
print(f'done: {(time.time()-t0)/60:.1f} min | {sum(len(v) for v in PRED.values())} events')

5517 test clips
  0/5517 | 0.0 min


/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


  500/5517 | 3.5 min
  1000/5517 | 4.6 min
  1500/5517 | 5.6 min
  2000/5517 | 6.7 min
  2500/5517 | 7.9 min
  3000/5517 | 9.0 min
  3500/5517 | 10.2 min
  4000/5517 | 11.4 min
  4500/5517 | 12.7 min
  5000/5517 | 14.0 min
  5500/5517 | 15.2 min
done: 15.3 min | 0 events


## 12 — Package submission

In [ ]:
recs=[
    {'clip_id':c,'events':[{'onset':round(float(a),3),'offset':round(float(b),3)} for a,b in PRED[c]]}
    for c in sorted(PRED)
]
ids=[r['clip_id'] for r in recs]
assert len(set(ids))==len(ids)==len(paths)
for r in recs:
    prev=-1.0
    for e in r['events']:
        assert 0<=e['onset']<e['offset']
        assert e['onset']>=prev-1e-6
        prev=e['offset']

jl=f'{WORK}/submission_boundary_WavLM_BEATs_verified_{RUN_TAG}.jsonl'
with open(jl,'w') as f:
    for r in recs: f.write(json.dumps(r)+'\n')
zp=f'{WORK}/submission_boundary_WavLM_BEATs_verified_{RUN_TAG}.zip'
with zipfile.ZipFile(zp,'w',zipfile.ZIP_DEFLATED) as z: z.write(jl,'predictions.jsonl')
assert zipfile.ZipFile(zp).namelist()==['predictions.jsonl']
print('SUBMIT:',zp)
print('encoder: WavLM Base+ + BEATs Iter3+ AS2M')
print('labels : verified_timestamps only')
print('snap   :',SNAP)

SUBMIT: /content/drive/MyDrive/Datathon/work/submission_boundary_WavLM_BEATs_verified_20260905T161244.zip
encoder: WavLM Base+ + BEATs Iter3+ AS2M
labels : verified_timestamps only
snap   : True


In [ ]:
# ============================================================
# OFFICIAL VALIDATION SET EVALUATION
# Best checkpoint -> released validationMetadata.json
#
# Reports:
#   - frame-level micro F1
#   - event P/R/F1 with notebook's relative 20% tolerance
#   - event F1 @ +/-100 ms
#   - event F1 @ +/-200 ms
#   - onset / offset MAE for matched events
#   - overall / natural / synthetic
#
# Does NOT permanently overwrite the current training model.
# ============================================================

import os
import json
import glob
import gc
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import torch


# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
VAL_ROOT = "/content/drive/MyDrive/PHD/Datathon/validation"

VAL_META = os.path.join(
    VAL_ROOT,
    "validationMetadata.json"
)

NAT_DIR = os.path.join(
    VAL_ROOT,
    "naturalNoisyAudio"
)

SYN_DIR = os.path.join(
    VAL_ROOT,
    "syntheticNoiseAudio"
)

CLEAN_DIR = os.path.join(
    VAL_ROOT,
    "syntheticCleanRefAudio"
)

BEST_PATH = (
    "/content/drive/MyDrive/Datathon/work/ckpt/"
    "boundary_wavlm_beats_verified_stable_best.pt"
)


assert os.path.isfile(VAL_META), VAL_META
assert os.path.isfile(BEST_PATH), BEST_PATH

print("Metadata   :", VAL_META)
print("Checkpoint :", BEST_PATH)


# ============================================================
# Load validation metadata
# ============================================================

with open(VAL_META, "r", encoding="utf-8") as f:
    val_meta = json.load(f)

print("\nValidation metadata rows:", len(val_meta))

quality_counts = {}

for r in val_meta:
    q = str(r.get("annotationQuality", "")).strip()
    quality_counts[q] = quality_counts.get(q, 0) + 1

print("Annotation quality:", quality_counts)


# ============================================================
# Index validation WAV files
#
# Prefer expected folder according to syntheticData, but make
# a basename index as a fallback.
# ============================================================

print("\nIndexing validation audio...")

audio_index = {}

# priority: actual noisy audio before clean reference
for folder in [NAT_DIR, SYN_DIR, CLEAN_DIR]:

    if not os.path.isdir(folder):
        print("WARNING: folder missing:", folder)
        continue

    paths = glob.glob(
        os.path.join(folder, "**", "*.wav"),
        recursive=True,
    )

    print(
        os.path.basename(folder),
        ":",
        len(paths),
        "wav files"
    )

    for p in paths:

        name = os.path.basename(p)

        # retain first occurrence according to priority above
        if name not in audio_index:
            audio_index[name] = p


print("Unique indexed names:", len(audio_index))


def resolve_validation_audio(row):

    fname = str(row["segmentFileName"])

    is_synth = bool(
        row.get("syntheticData", False)
    )

    # first try exact expected directory
    primary = SYN_DIR if is_synth else NAT_DIR

    p = os.path.join(primary, fname)

    if os.path.isfile(p):
        return p

    # basename fallback
    base = os.path.basename(fname)

    if base in audio_index:
        return audio_index[base]

    return None


# ============================================================
# Parse ground-truth intervals
# ============================================================

def get_gt_events(row):

    raw = row.get(
        "NoiseSubCategoryTimeStamp",
        []
    )

    if not isinstance(raw, list):
        return []

    try:
        dur = float(row["duration"])
    except Exception:
        dur = None

    events = []

    for ev in raw:

        if not isinstance(ev, dict):
            continue

        try:
            s = float(ev["start"])
            e = float(ev["end"])
        except Exception:
            continue

        if not np.isfinite(s) or not np.isfinite(e):
            continue

        s = max(0.0, s)

        if dur is not None:
            s = min(s, dur)
            e = min(e, dur)

        if e > s:
            events.append((s, e))

    return events


# ============================================================
# Audio loader — identical signal path to training
# ============================================================

def load_val_audio(path):

    y, sr0 = sf.read(
        path,
        dtype="float32",
        always_2d=False,
    )

    y = np.asarray(
        y,
        dtype=np.float32
    )

    if y.ndim == 2:
        y = y.mean(
            axis=1 if y.shape[0] >= y.shape[1] else 0
        )

    y = y.squeeze()

    if sr0 != SR:
        y = librosa.resample(
            y,
            orig_sr=sr0,
            target_sr=SR,
        ).astype(np.float32)

    y = np.nan_to_num(
        y,
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
    ).astype(np.float32)

    return y


# ============================================================
# Decoder
#
# Same defaults used in our notebook.
# ============================================================

DECODER = {
    "thr": 0.85,
    "med_ms": 200,
    "min_dur": 0.12,
    "merge_gap": 0.10,
}

SNAP_WINDOW = 0.20


# Define fallbacks in case decoder cells were not executed.
if "decode_standard" not in globals():

    from scipy.ndimage import median_filter

    def decode_standard(
        p,
        dur,
        thr=0.85,
        med_ms=200,
        min_dur=0.12,
        merge_gap=0.10,
    ):

        m = p >= thr

        k = max(
            1,
            int(round(
                (med_ms / 1000.0) / FRAME
            ))
        )

        if k > 1:

            if k % 2 == 0:
                k += 1

            m = (
                median_filter(
                    m.astype(np.float32),
                    size=k,
                )
                > 0.5
            )

        events = []
        start = None

        for i, v in enumerate(m):

            if v and start is None:
                start = i

            elif not v and start is not None:

                events.append([
                    start * FRAME,
                    i * FRAME,
                ])

                start = None

        if start is not None:
            events.append([
                start * FRAME,
                len(m) * FRAME,
            ])

        merged = []

        for e in events:

            if (
                merged
                and e[0] - merged[-1][1]
                <= merge_gap
            ):
                merged[-1][1] = e[1]

            else:
                merged.append(e)

        return [
            (a, min(b, dur))
            for a, b in merged
            if min(b, dur) - a >= min_dur
        ]


if "snap_boundaries" not in globals():

    def snap_boundaries(
        candidates,
        onset_dist,
        offset_dist,
        dur,
        snap_window=0.20,
        min_dur=0.12,
    ):

        out = []

        W = max(
            1,
            int(round(
                snap_window / FRAME
            ))
        )

        N = len(onset_dist)

        for onset, offset in candidates:

            on_f = int(
                round(onset / FRAME)
            )

            off_f = int(
                round(offset / FRAME)
            )

            a = max(0, on_f - W)
            b = min(N, on_f + W + 1)

            if b > a:
                best_on = (
                    a
                    + int(
                        np.argmin(
                            onset_dist[a:b]
                        )
                    )
                ) * FRAME
            else:
                best_on = onset

            a = max(0, off_f - W)
            b = min(N, off_f + W + 1)

            if b > a:
                best_off = (
                    a
                    + int(
                        np.argmin(
                            offset_dist[a:b]
                        )
                    )
                ) * FRAME
            else:
                best_off = offset

            best_on = max(
                0.0,
                best_on
            )

            best_off = min(
                float(dur),
                best_off
            )

            if best_off - best_on >= min_dur:
                out.append(
                    (best_on, best_off)
                )

        return out


# ============================================================
# Event matching
# ============================================================

def match_events(
    pred,
    ref,
    mode="fixed",
    tolerance=0.2,
):

    """
    Greedy one-to-one matching.

    mode='fixed':
        onset AND offset must each be within
        tolerance seconds.

    mode='relative':
        tolerance = tolerance * reference duration.
    """

    matched_ref = set()
    matches = []

    for pi, (po, pf) in enumerate(pred):

        candidates = []

        for ri, (ro, rf) in enumerate(ref):

            if ri in matched_ref:
                continue

            if mode == "relative":
                t = tolerance * max(
                    rf - ro,
                    1e-6,
                )
            else:
                t = tolerance

            on_err = abs(po - ro)
            off_err = abs(pf - rf)

            if on_err <= t and off_err <= t:

                candidates.append(
                    (
                        on_err + off_err,
                        ri,
                        on_err,
                        off_err,
                    )
                )

        if candidates:

            candidates.sort(
                key=lambda x: x[0]
            )

            _, ri, on_err, off_err = (
                candidates[0]
            )

            matched_ref.add(ri)

            matches.append(
                (
                    pi,
                    ri,
                    on_err,
                    off_err,
                )
            )

    tp = len(matches)
    fp = len(pred) - tp
    fn = len(ref) - tp

    return tp, fp, fn, matches


def prf(tp, fp, fn):

    precision = tp / max(
        tp + fp,
        1,
    )

    recall = tp / max(
        tp + fn,
        1,
    )

    f1 = (
        2 * precision * recall
        / max(
            precision + recall,
            1e-12,
        )
    )

    return precision, recall, f1


# ============================================================
# Frame-level metric
# ============================================================

def intervals_to_mask(
    events,
    n_frames,
):

    m = np.zeros(
        n_frames,
        dtype=np.bool_,
    )

    for s, e in events:

        a = max(
            0,
            min(
                int(np.floor(s / FRAME)),
                n_frames,
            )
        )

        b = max(
            a,
            min(
                int(np.ceil(e / FRAME)),
                n_frames,
            )
        )

        m[a:b] = True

    return m


# ============================================================
# Load BEST checkpoint
#
# Backup only checkpoint-overlapping current weights, so if
# you're continuing training afterwards they can be restored.
# ============================================================

assert "model" in globals(), (
    "Run the BoundaryModel/model construction cell first."
)

print("\nBacking up current trainable state...")

model_state_now = model.state_dict()

current_backup = {}


try:
    saved = torch.load(
        BEST_PATH,
        map_location="cpu",
        weights_only=False,
    )
except TypeError:
    saved = torch.load(
        BEST_PATH,
        map_location="cpu",
    )


ckpt_state = saved["model"]


for k in ckpt_state:

    if k in model_state_now:
        current_backup[k] = (
            model_state_now[k]
            .detach()
            .cpu()
            .clone()
        )


missing, unexpected = model.load_state_dict(
    ckpt_state,
    strict=False,
)


bad_missing = [
    k for k in missing
    if not (
        k.startswith("wavlm.m.")
        or k.startswith("beats.m.")
        or k.startswith("wavlm.")
        or k.startswith("beats.")
    )
]

if bad_missing or unexpected:

    print("WARNING checkpoint load:")
    print(" unexpected:", unexpected[:10])
    print(" missing   :", bad_missing[:10])


model.eval()

print(
    "\nLoaded checkpoint epoch:",
    saved.get("epoch", "?")
)

print(
    "Internal best val loss:",
    saved.get("best_val_loss", "?")
)


# ============================================================
# Metric accumulators
# ============================================================

groups = {
    "overall": [],
    "natural": [],
    "synthetic": [],
}

missing_files = []
failed_files = []

results = []


# ============================================================
# INFERENCE
# ============================================================

print("\nRunning official validation inference...\n")

with torch.inference_mode():

    for i, row in enumerate(
        val_meta,
        start=1,
    ):

        # ---------------------------------------------
        # Only timestamps we can objectively evaluate
        # ---------------------------------------------
        quality = str(
            row.get(
                "annotationQuality",
                ""
            )
        ).strip().lower()

        if quality != "verified_timestamps":
            continue


        gt = get_gt_events(row)

        # If there is no timestamp ground truth,
        # boundary metrics cannot be calculated.
        if len(gt) == 0:
            continue


        path = resolve_validation_audio(
            row
        )

        if path is None:

            missing_files.append(
                row["segmentFileName"]
            )

            continue


        try:

            # -----------------------------------------
            # waveform
            # -----------------------------------------
            y = load_val_audio(path)

            valid_samples = len(y)

            dur_audio = (
                valid_samples / SR
            )


            # -----------------------------------------
            # exact training mel pipeline
            # -----------------------------------------
            mel = logmel(y)

            mel = (
                mel - mel.mean()
            ) / (
                mel.std() + 1e-5
            )

            mel = np.asarray(
                mel,
                dtype=np.float32,
            )

            T = mel.shape[1]


            # -----------------------------------------
            # waveform aligned to mel timeline
            # -----------------------------------------
            need = T * HOP

            yy = np.pad(
                y,
                (
                    0,
                    max(
                        0,
                        need - valid_samples,
                    ),
                ),
            )[:need].astype(
                np.float32
            )


            xt = (
                torch
                .from_numpy(mel)
                .unsqueeze(0)
                .unsqueeze(0)
                .to(
                    device,
                    dtype=torch.float32,
                )
            )

            wt = (
                torch
                .from_numpy(yy)
                .unsqueeze(0)
                .to(
                    device,
                    dtype=torch.float32,
                )
            )

            wl = torch.tensor(
                [
                    min(
                        valid_samples,
                        need,
                    )
                ],
                dtype=torch.long,
                device=device,
            )


            # -----------------------------------------
            # model
            # -----------------------------------------
            p_n, p_o, p_f = model(
                xt,
                wt,
                wl,
            )


            if not (
                torch.isfinite(p_n).all()
                and torch.isfinite(p_o).all()
                and torch.isfinite(p_f).all()
            ):
                raise FloatingPointError(
                    "non-finite model output"
                )


            pn = (
                torch.sigmoid(
                    p_n[0, :T]
                )
                .float()
                .cpu()
                .numpy()
            )

            po = (
                p_o[0, :T]
                .float()
                .cpu()
                .numpy()
            )

            pf = (
                p_f[0, :T]
                .float()
                .cpu()
                .numpy()
            )


            # -----------------------------------------
            # decode
            # -----------------------------------------
            candidates = decode_standard(
                pn,
                dur_audio,
                **DECODER,
            )

            snapped = snap_boundaries(
                candidates,
                po,
                pf,
                dur_audio,
                snap_window=SNAP_WINDOW,
            )


            is_synth = bool(
                row.get(
                    "syntheticData",
                    False
                )
            )

            split = (
                "synthetic"
                if is_synth
                else "natural"
            )


            rec = {
                "file":
                    row["segmentFileName"],

                "split":
                    split,

                "language":
                    row.get(
                        "language",
                        ""
                    ),

                "duration":
                    dur_audio,

                "gt":
                    gt,

                "pred_raw":
                    candidates,

                "pred_snap":
                    snapped,

                "pn":
                    pn,
            }


            results.append(rec)

            groups["overall"].append(rec)
            groups[split].append(rec)


        except Exception as e:

            failed_files.append(
                (
                    row["segmentFileName"],
                    type(e).__name__,
                    str(e),
                )
            )


        if i % 100 == 0:

            print(
                f"{i}/{len(val_meta)} "
                f"| evaluated={len(results)} "
                f"| missing={len(missing_files)} "
                f"| failed={len(failed_files)}",
                flush=True,
            )


# ============================================================
# Score one group
# ============================================================

def score_group(records, use_snap=True):

    totals = {
        "rel": [0, 0, 0],
        "100": [0, 0, 0],
        "200": [0, 0, 0],
    }

    onset_errs = []
    offset_errs = []

    frame_tp = 0
    frame_fp = 0
    frame_fn = 0

    macro_f1_200 = []


    for r in records:

        pred = (
            r["pred_snap"]
            if use_snap
            else r["pred_raw"]
        )

        ref = r["gt"]


        # -----------------------------------------
        # notebook's original metric
        # +/- 20% reference event duration
        # -----------------------------------------
        tp, fp, fn, _ = match_events(
            pred,
            ref,
            mode="relative",
            tolerance=0.20,
        )

        totals["rel"][0] += tp
        totals["rel"][1] += fp
        totals["rel"][2] += fn


        # -----------------------------------------
        # fixed +/-100 ms
        # -----------------------------------------
        tp, fp, fn, _ = match_events(
            pred,
            ref,
            mode="fixed",
            tolerance=0.100,
        )

        totals["100"][0] += tp
        totals["100"][1] += fp
        totals["100"][2] += fn


        # -----------------------------------------
        # fixed +/-200 ms
        # -----------------------------------------
        tp, fp, fn, matches = (
            match_events(
                pred,
                ref,
                mode="fixed",
                tolerance=0.200,
            )
        )

        totals["200"][0] += tp
        totals["200"][1] += fp
        totals["200"][2] += fn


        p, rc, f = prf(
            tp,
            fp,
            fn,
        )

        macro_f1_200.append(f)


        for (
            _pi,
            _ri,
            on_err,
            off_err,
        ) in matches:

            onset_errs.append(
                on_err
            )

            offset_errs.append(
                off_err
            )


        # -----------------------------------------
        # frame-level noise F1
        # -----------------------------------------
        n_frames = len(
            r["pn"]
        )

        pm = intervals_to_mask(
            pred,
            n_frames,
        )

        gm = intervals_to_mask(
            ref,
            n_frames,
        )

        frame_tp += int(
            np.logical_and(
                pm,
                gm,
            ).sum()
        )

        frame_fp += int(
            np.logical_and(
                pm,
                ~gm,
            ).sum()
        )

        frame_fn += int(
            np.logical_and(
                ~pm,
                gm,
            ).sum()
        )


    out = {
        "clips": len(records)
    }


    for key, name in [
        ("rel", "rel20"),
        ("100", "100ms"),
        ("200", "200ms"),
    ]:

        tp, fp, fn = totals[key]

        p, r, f = prf(
            tp,
            fp,
            fn,
        )

        out[f"P_{name}"] = p
        out[f"R_{name}"] = r
        out[f"F1_{name}"] = f


    fp_, fr_, ff_ = prf(
        frame_tp,
        frame_fp,
        frame_fn,
    )

    out["frame_P"] = fp_
    out["frame_R"] = fr_
    out["frame_F1"] = ff_


    out["macro_F1_200ms"] = (
        float(
            np.mean(
                macro_f1_200
            )
        )
        if macro_f1_200
        else np.nan
    )


    out["onset_MAE_ms"] = (
        1000
        * float(
            np.mean(
                onset_errs
            )
        )
        if onset_errs
        else np.nan
    )

    out["offset_MAE_ms"] = (
        1000
        * float(
            np.mean(
                offset_errs
            )
        )
        if offset_errs
        else np.nan
    )


    return out


# ============================================================
# Compare RAW vs boundary snapping
# ============================================================

rows = []

for group_name in [
    "overall",
    "natural",
    "synthetic",
]:

    records = groups[
        group_name
    ]

    if not records:
        continue

    for snap in [
        False,
        True,
    ]:

        score = score_group(
            records,
            use_snap=snap,
        )

        rows.append({
            "group":
                group_name,

            "decoder":
                (
                    "snap"
                    if snap
                    else "no_snap"
                ),

            **score,
        })


score_df = pd.DataFrame(rows)


# ============================================================
# Results
# ============================================================

print("\n")
print("=" * 80)
print("OFFICIAL VALIDATION RESULTS")
print("=" * 80)

cols = [
    "group",
    "decoder",
    "clips",
    "frame_F1",
    "F1_rel20",
    "F1_100ms",
    "F1_200ms",
    "onset_MAE_ms",
    "offset_MAE_ms",
]

print(
    score_df[
        cols
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}",
    )
)


print("\nDecoder parameters:")
print(DECODER)

print(
    "snap window:",
    SNAP_WINDOW,
    "sec"
)

print(
    "\nSuccessfully evaluated:",
    len(results)
)

print(
    "Missing audio files:",
    len(missing_files)
)

print(
    "Failed files:",
    len(failed_files)
)


if missing_files:

    print(
        "\nFirst missing files:"
    )

    for x in missing_files[:5]:
        print(" ", x)


if failed_files:

    print(
        "\nFirst failed files:"
    )

    for x in failed_files[:5]:
        print(" ", x)


# ============================================================
# Show several predictions
# ============================================================

print("\nExample predictions:")

for r in results[:5]:

    print(
        "\n",
        r["file"]
    )

    print(
        " split:",
        r["split"]
    )

    print(
        " GT :",
        [
            (
                round(a, 3),
                round(b, 3),
            )
            for a, b in r["gt"]
        ]
    )

    print(
        " RAW:",
        [
            (
                round(a, 3),
                round(b, 3),
            )
            for a, b in r["pred_raw"]
        ]
    )

    print(
        " SNAP:",
        [
            (
                round(a, 3),
                round(b, 3),
            )
            for a, b in r["pred_snap"]
        ]
    )


# ============================================================
# Restore the model that was in memory before validation
#
# This lets you continue training from the current epoch rather
# than accidentally continuing from the older best checkpoint.
# ============================================================

print(
    "\nRestoring pre-validation model weights..."
)

model.load_state_dict(
    current_backup,
    strict=False,
)

print(
    "Current training model restored."
)

model.train()

if hasattr(model, "wavlm"):
    model.wavlm.eval()

if hasattr(model, "beats"):
    model.beats.eval()


del current_backup
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\nValidation complete.")

Metadata   : /content/drive/MyDrive/PHD/Datathon/validation/validationMetadata.json
Checkpoint : /content/drive/MyDrive/Datathon/work/ckpt/boundary_wavlm_beats_verified_stable_best.pt

Validation metadata rows: 2501
Annotation quality: {'verified_timestamps': 2501}

Indexing validation audio...
naturalNoisyAudio : 1806 wav files
syntheticNoiseAudio : 695 wav files
syntheticCleanRefAudio : 695 wav files
Unique indexed names: 2501

Backing up current trainable state...

Loaded checkpoint epoch: 2
Internal best val loss: 0.40138038700983464

Running official validation inference...

100/2501 | evaluated=100 | missing=0 | failed=0
200/2501 | evaluated=200 | missing=0 | failed=0
300/2501 | evaluated=300 | missing=0 | failed=0
400/2501 | evaluated=400 | missing=0 | failed=0
500/2501 | evaluated=500 | missing=0 | failed=0
600/2501 | evaluated=600 | missing=0 | failed=0
700/2501 | evaluated=700 | missing=0 | failed=0
800/2501 | evaluated=800 | missing=0 | failed=0
900/2501 | evaluated=900 | mi

KeyboardInterrupt: 

In [ ]:
# ============================================================
# AUTO-TUNE DECODER ON RELEASED VALIDATION SET
# + GENERATE TUNED predictions.jsonl
# ============================================================

import os
import json
import glob
import pickle
import zipfile
import itertools
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import torch

from scipy.ndimage import median_filter


# ============================================================
# PATHS
# ============================================================

VAL_ROOT = "/content/drive/MyDrive/PHD/Datathon/validation"

VAL_META = os.path.join(
    VAL_ROOT,
    "validationMetadata.json"
)

NAT_DIR = os.path.join(
    VAL_ROOT,
    "naturalNoisyAudio"
)

SYN_DIR = os.path.join(
    VAL_ROOT,
    "syntheticNoiseAudio"
)

BEST_PATH = (
    "/content/drive/MyDrive/Datathon/work/ckpt/"
    "boundary_wavlm_beats_verified_stable_best.pt"
)

OUT_DIR = "/content/drive/MyDrive/Datathon/work"
os.makedirs(OUT_DIR, exist_ok=True)

assert os.path.isfile(VAL_META)
assert os.path.isfile(BEST_PATH)

print("Validation:", VAL_ROOT)
print("Checkpoint:", BEST_PATH)


# ============================================================
# LOAD BEST CHECKPOINT
# ============================================================

try:
    saved = torch.load(
        BEST_PATH,
        map_location="cpu",
        weights_only=False
    )
except TypeError:
    saved = torch.load(
        BEST_PATH,
        map_location="cpu"
    )


missing, unexpected = model.load_state_dict(
    saved["model"],
    strict=False
)

model.eval()

BEST_EPOCH = saved.get("epoch", "unknown")

print("\nLoaded best checkpoint")
print("epoch:", BEST_EPOCH)
print(
    "internal val loss:",
    saved.get("best_val_loss", "?")
)


# ============================================================
# METADATA
# ============================================================

with open(
    VAL_META,
    "r",
    encoding="utf-8"
) as f:

    metadata = json.load(f)


print(
    "\nValidation clips:",
    len(metadata)
)


# ============================================================
# INDEX ACTUAL NOISY AUDIO
#
# IMPORTANT:
# syntheticCleanRefAudio is NOT used as model input.
# ============================================================

nat_index = {}
syn_index = {}

for p in glob.glob(
    os.path.join(
        NAT_DIR,
        "**",
        "*.wav"
    ),
    recursive=True
):

    nat_index[
        os.path.basename(p)
    ] = p


for p in glob.glob(
    os.path.join(
        SYN_DIR,
        "**",
        "*.wav"
    ),
    recursive=True
):

    syn_index[
        os.path.basename(p)
    ] = p


print(
    "Natural WAVs :",
    len(nat_index)
)

print(
    "Synthetic WAVs:",
    len(syn_index)
)


def resolve_path(row):

    fname = os.path.basename(
        row["segmentFileName"]
    )

    is_synthetic = bool(
        row.get(
            "syntheticData",
            False
        )
    )

    if is_synthetic:

        return syn_index.get(fname)

    return nat_index.get(fname)


# ============================================================
# GROUND TRUTH
# ============================================================

def get_gt(row):

    events = []

    raw = row.get(
        "NoiseSubCategoryTimeStamp",
        []
    )

    if not isinstance(raw, list):
        return events

    try:
        dur = float(
            row["duration"]
        )
    except Exception:
        dur = None


    for e in raw:

        try:

            s = float(
                e["start"]
            )

            f = float(
                e["end"]
            )

        except Exception:
            continue


        s = max(
            0.0,
            s
        )

        if dur is not None:

            s = min(
                s,
                dur
            )

            f = min(
                f,
                dur
            )


        if f > s:

            events.append(
                (
                    s,
                    f
                )
            )

    return events


# ============================================================
# AUDIO
# ============================================================

def load_audio(path):

    y, sr = sf.read(
        path,
        dtype="float32",
        always_2d=False
    )


    y = np.asarray(
        y,
        dtype=np.float32
    )


    if y.ndim == 2:

        y = y.mean(
            axis=1
        )


    if sr != SR:

        y = librosa.resample(
            y,
            orig_sr=sr,
            target_sr=SR
        ).astype(
            np.float32
        )


    return np.nan_to_num(
        y,
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    ).astype(
        np.float32
    )


# ============================================================
# RUN MODEL ONCE AND CACHE POSTERIORS
# ============================================================

CACHE_PATH = os.path.join(
    OUT_DIR,
    f"official_val_posteriors_epoch_{BEST_EPOCH}.pkl"
)


if os.path.isfile(CACHE_PATH):

    print(
        "\nLoading cached validation inference..."
    )

    with open(
        CACHE_PATH,
        "rb"
    ) as f:

        records = pickle.load(f)

else:

    print(
        "\nRunning model over official validation..."
    )

    records = []

    missing_audio = []


    model.eval()


    with torch.inference_mode():

        for i, row in enumerate(
            metadata
        ):

            path = resolve_path(
                row
            )


            if path is None:

                missing_audio.append(
                    row["segmentFileName"]
                )

                continue


            y = load_audio(
                path
            )


            valid_samples = len(y)

            dur = (
                valid_samples
                / float(SR)
            )


            # ----------------------------------------
            # Same mel pipeline as training
            # ----------------------------------------

            mel = logmel(y)

            mel = (
                mel - mel.mean()
            ) / (
                mel.std() + 1e-5
            )


            T = mel.shape[1]


            need = T * HOP


            yy = np.pad(
                y,
                (
                    0,
                    max(
                        0,
                        need - valid_samples
                    )
                )
            )[:need].astype(
                np.float32
            )


            xt = (
                torch
                .from_numpy(mel)
                [None, None]
                .float()
                .to(device)
            )


            wt = (
                torch
                .from_numpy(yy)
                [None]
                .float()
                .to(device)
            )


            wl = torch.tensor(
                [
                    min(
                        valid_samples,
                        need
                    )
                ],
                dtype=torch.long,
                device=device
            )


            p_n, p_on, p_off = model(
                xt,
                wt,
                wl
            )


            prob = (
                torch.sigmoid(
                    p_n[0]
                )
                .float()
                .cpu()
                .numpy()
            )


            onset_dist = (
                p_on[0]
                .float()
                .cpu()
                .numpy()
            )


            offset_dist = (
                p_off[0]
                .float()
                .cpu()
                .numpy()
            )


            clip_id = os.path.splitext(
                os.path.basename(
                    row["segmentFileName"]
                )
            )[0]


            records.append({

                "clip_id":
                    clip_id,

                "duration":
                    dur,

                "prob":
                    prob,

                "onset_dist":
                    onset_dist,

                "offset_dist":
                    offset_dist,

                "gt":
                    get_gt(row),

                "synthetic":
                    bool(
                        row.get(
                            "syntheticData",
                            False
                        )
                    ),

                "language":
                    row.get(
                        "language",
                        ""
                    ),
            })


            if (
                i == 0
                or (i + 1) % 250 == 0
            ):

                print(
                    f"{i+1}/{len(metadata)}",
                    flush=True
                )


    print(
        "missing audio:",
        len(missing_audio)
    )


    if missing_audio:

        print(
            "first missing:",
            missing_audio[:5]
        )


    with open(
        CACHE_PATH,
        "wb"
    ) as f:

        pickle.dump(
            records,
            f,
            protocol=pickle.HIGHEST_PROTOCOL
        )


    print(
        "Cached:",
        CACHE_PATH
    )


print(
    "\nInference records:",
    len(records)
)


# ============================================================
# POSTERIOR DIAGNOSTIC
# ============================================================

all_probs = np.concatenate(
    [
        r["prob"]
        for r in records
    ]
)


print("\nPOSTERIOR DISTRIBUTION")
print("======================")

for q in [
    0,
    10,
    25,
    50,
    75,
    90,
    95,
    99,
    100
]:

    print(
        f"p{q:03d}: "
        f"{np.percentile(all_probs,q):.4f}"
    )


# ============================================================
# MASK HELPER
# ============================================================

def events_to_mask(
    events,
    N
):

    m = np.zeros(
        N,
        dtype=bool
    )


    for s, e in events:

        a = max(
            0,
            min(
                int(
                    np.floor(
                        s / FRAME
                    )
                ),
                N
            )
        )


        b = max(
            a,
            min(
                int(
                    np.ceil(
                        e / FRAME
                    )
                ),
                N
            )
        )


        m[a:b] = True


    return m


# ============================================================
# STAGE 1:
# FIND ROUGH THRESHOLD BY FRAME DICE
# ============================================================

THRESHOLDS = np.arange(
    0.05,
    0.805,
    0.025
)


threshold_results = []


for thr in THRESHOLDS:

    tp = fp = fn = 0


    for r in records:

        pred = (
            r["prob"]
            >= thr
        )


        gt = events_to_mask(
            r["gt"],
            len(pred)
        )


        tp += np.logical_and(
            pred,
            gt
        ).sum()


        fp += np.logical_and(
            pred,
            ~gt
        ).sum()


        fn += np.logical_and(
            ~pred,
            gt
        ).sum()


    dice = (
        2 * tp
        / max(
            2 * tp + fp + fn,
            1
        )
    )


    threshold_results.append(
        (
            float(thr),
            float(dice)
        )
    )


threshold_results.sort(
    key=lambda x: x[1],
    reverse=True
)


print("\nBEST RAW THRESHOLDS")
print("===================")

for t, d in threshold_results[:10]:

    print(
        f"thr={t:.3f} "
        f"frameDice={d:.4f}"
    )


ROUGH_THR = threshold_results[0][0]


print(
    "\nRough best threshold:",
    ROUGH_THR
)


# ============================================================
# DECODER
# ============================================================

def decode(
    p,
    dur,
    thr,
    med_ms,
    min_dur,
    merge_gap
):

    mask = (
        p >= thr
    )


    # ----------------------------------------
    # median smoothing
    # ----------------------------------------

    if med_ms > 0:

        k = max(
            1,
            int(
                round(
                    (
                        med_ms / 1000
                    )
                    / FRAME
                )
            )
        )


        if k % 2 == 0:
            k += 1


        if k > 1:

            mask = (
                median_filter(
                    mask.astype(
                        np.float32
                    ),
                    size=k
                )
                > 0.5
            )


    # ----------------------------------------
    # mask -> intervals
    # ----------------------------------------

    events = []

    start = None


    for i, v in enumerate(
        mask
    ):

        if v and start is None:

            start = i


        elif (
            not v
            and start is not None
        ):

            events.append(
                [
                    start * FRAME,
                    i * FRAME
                ]
            )

            start = None


    if start is not None:

        events.append(
            [
                start * FRAME,
                len(mask) * FRAME
            ]
        )


    # ----------------------------------------
    # merge
    # ----------------------------------------

    merged = []


    for e in events:

        if (
            merged
            and
            e[0] - merged[-1][1]
            <= merge_gap
        ):

            merged[-1][1] = e[1]

        else:

            merged.append(e)


    # ----------------------------------------
    # min duration
    # ----------------------------------------

    return [

        (
            max(
                0.0,
                float(a)
            ),

            min(
                float(b),
                float(dur)
            )
        )

        for a, b in merged

        if (
            min(
                float(b),
                float(dur)
            )
            - float(a)
            >= min_dur
        )
    ]


# ============================================================
# OPTIONAL BOUNDARY SNAPPING
# ============================================================

def snap(
    candidates,
    onset_dist,
    offset_dist,
    dur,
    window=0.20,
    min_dur=0.03
):

    N = len(
        onset_dist
    )


    W = max(
        1,
        int(
            round(
                window / FRAME
            )
        )
    )


    out = []


    for onset, offset in candidates:

        of = int(
            round(
                onset / FRAME
            )
        )

        ff = int(
            round(
                offset / FRAME
            )
        )


        # onset
        a = max(
            0,
            of - W
        )

        b = min(
            N,
            of + W + 1
        )


        best_on = onset


        if b > a:

            best_on = (
                a
                + int(
                    np.argmin(
                        onset_dist[a:b]
                    )
                )
            ) * FRAME


        # offset
        a = max(
            0,
            ff - W
        )

        b = min(
            N,
            ff + W + 1
        )


        best_off = offset


        if b > a:

            best_off = (
                a
                + int(
                    np.argmin(
                        offset_dist[a:b]
                    )
                )
            ) * FRAME


        best_on = max(
            0.0,
            best_on
        )


        best_off = min(
            float(dur),
            best_off
        )


        if (
            best_off
            - best_on
            >= min_dur
        ):

            out.append(
                (
                    best_on,
                    best_off
                )
            )


    return out


# ============================================================
# EVENT MATCHING
#
# NOTE:
# This is a proxy metric using +/-200 ms boundary tolerance.
# Use the competition scorer afterwards for the authoritative
# score.
# ============================================================

def event_counts(
    pred,
    ref,
    tol=0.20
):

    matched = set()

    tp = 0


    for po, pf in pred:

        best = None


        for j, (
            ro,
            rf
        ) in enumerate(ref):

            if j in matched:
                continue


            d = (
                abs(
                    po - ro
                )
                +
                abs(
                    pf - rf
                )
            )


            if (
                abs(
                    po - ro
                ) <= tol
                and
                abs(
                    pf - rf
                ) <= tol
            ):

                if (
                    best is None
                    or d < best[0]
                ):

                    best = (
                        d,
                        j
                    )


        if best is not None:

            matched.add(
                best[1]
            )

            tp += 1


    fp = (
        len(pred)
        - tp
    )

    fn = (
        len(ref)
        - tp
    )


    return (
        tp,
        fp,
        fn
    )


# ============================================================
# SCORE DECODER PARAMETERS
# ============================================================

def score_records(
    subset,
    params
):

    (
        thr,
        med_ms,
        min_dur,
        merge_gap,
        use_snap
    ) = params


    ev_tp = ev_fp = ev_fn = 0

    seg_tp = seg_fp = seg_fn = 0


    for r in subset:

        pred = decode(
            r["prob"],
            r["duration"],
            thr,
            med_ms,
            min_dur,
            merge_gap
        )


        if use_snap:

            pred = snap(
                pred,
                r["onset_dist"],
                r["offset_dist"],
                r["duration"],
                min_dur=min_dur
            )


        tp, fp, fn = event_counts(
            pred,
            r["gt"],
            tol=0.20
        )


        ev_tp += tp
        ev_fp += fp
        ev_fn += fn


        pm = events_to_mask(
            pred,
            len(
                r["prob"]
            )
        )


        gm = events_to_mask(
            r["gt"],
            len(
                r["prob"]
            )
        )


        seg_tp += np.logical_and(
            pm,
            gm
        ).sum()


        seg_fp += np.logical_and(
            pm,
            ~gm
        ).sum()


        seg_fn += np.logical_and(
            ~pm,
            gm
        ).sum()


    precision = (
        ev_tp
        / max(
            ev_tp + ev_fp,
            1
        )
    )


    recall = (
        ev_tp
        / max(
            ev_tp + ev_fn,
            1
        )
    )


    event_f1 = (
        2
        * precision
        * recall
        / max(
            precision + recall,
            1e-12
        )
    )


    dice = (
        2
        * seg_tp
        / max(
            2 * seg_tp
            + seg_fp
            + seg_fn,
            1
        )
    )


    combined = (
        event_f1
        + dice
    ) / 2


    return (
        float(combined),
        float(event_f1),
        float(dice)
    )


# ============================================================
# STAGE 2:
# PARAMETER SEARCH
#
# Use random 1500 clips for speed.
# Final best setting is then evaluated on ALL validation clips.
# ============================================================

rng = np.random.RandomState(
    42
)


TUNE_N = min(
    1500,
    len(records)
)


idx = rng.choice(
    len(records),
    size=TUNE_N,
    replace=False
)


tune_records = [
    records[i]
    for i in idx
]


candidate_thresholds = sorted(
    set(
        [
            round(
                float(
                    np.clip(
                        ROUGH_THR + d,
                        0.03,
                        0.90
                    )
                ),
                3
            )

            for d in [
                -0.10,
                -0.05,
                0.0,
                0.05,
                0.10
            ]
        ]
    )
)


MED_WINDOWS = [
    0,
    50,
    100
]


MIN_DURS = [
    0.03,
    0.06,
    0.10
]


MERGE_GAPS = [
    0.0,
    0.03,
    0.06
]


SNAP_OPTIONS = [
    False,
    True
]


grid = list(
    itertools.product(
        candidate_thresholds,
        MED_WINDOWS,
        MIN_DURS,
        MERGE_GAPS,
        SNAP_OPTIONS
    )
)


print(
    "\nSearching",
    len(grid),
    "decoder settings on",
    TUNE_N,
    "clips..."
)


search_results = []


for i, params in enumerate(
    grid
):

    score = score_records(
        tune_records,
        params
    )


    search_results.append(
        (
            score,
            params
        )
    )


    if (
        (i + 1) % 25
        == 0
    ):

        print(
            f"{i+1}/{len(grid)}",
            flush=True
        )


search_results.sort(
    key=lambda x: x[0][0],
    reverse=True
)


print("\nTOP DECODER SETTINGS")
print("====================")


for (
    score,
    params
) in search_results[:10]:

    (
        combined,
        event_f1,
        dice
    ) = score


    (
        thr,
        med,
        md,
        mg,
        sn
    ) = params


    print(
        f"combined={combined:.4f} "
        f"F1={event_f1:.4f} "
        f"Dice={dice:.4f} | "
        f"thr={thr:.3f} "
        f"med={med}ms "
        f"min={md:.3f}s "
        f"merge={mg:.3f}s "
        f"snap={sn}"
    )


BEST_PARAMS = (
    search_results[0][1]
)


print(
    "\nBEST PARAMS:",
    BEST_PARAMS
)


# ============================================================
# FINAL SCORE ON ALL RELEASED VALIDATION
# ============================================================

combined, ef1, dice = (
    score_records(
        records,
        BEST_PARAMS
    )
)


print("\nALL VALIDATION")
print("==============")

print(
    f"Proxy Event F1 : {ef1:.4f}"
)

print(
    f"Segment Dice   : {dice:.4f}"
)

print(
    f"Proxy Combined : {combined:.4f}"
)


# ============================================================
# NATURAL VS SYNTHETIC
# ============================================================

natural = [
    r
    for r in records
    if not r["synthetic"]
]

synthetic = [
    r
    for r in records
    if r["synthetic"]
]


for name, subset in [
    (
        "Natural",
        natural
    ),
    (
        "Synthetic",
        synthetic
    )
]:

    if not subset:
        continue


    c, f, d = score_records(
        subset,
        BEST_PARAMS
    )


    print(
        f"{name:10s} | "
        f"F1={f:.4f} "
        f"Dice={d:.4f} "
        f"Combined={c:.4f} "
        f"clips={len(subset)}"
    )


# ============================================================
# CREATE TUNED PREDICTIONS
# ============================================================

(
    best_thr,
    best_med,
    best_min,
    best_merge,
    best_snap
) = BEST_PARAMS


recs = []


total_events = 0

clips_with_events = 0


for r in records:

    pred = decode(
        r["prob"],
        r["duration"],
        best_thr,
        best_med,
        best_min,
        best_merge
    )


    if best_snap:

        pred = snap(
            pred,
            r["onset_dist"],
            r["offset_dist"],
            r["duration"],
            min_dur=best_min
        )


    # sort and basic safety
    pred = sorted(
        pred,
        key=lambda x: x[0]
    )


    clean_pred = []

    previous_offset = -1.0


    for a, b in pred:

        a = max(
            0.0,
            float(a)
        )

        b = min(
            float(r["duration"]),
            float(b)
        )


        if b <= a:
            continue


        # Prevent overlapping output events
        if a < previous_offset:

            a = previous_offset


        if b <= a:
            continue


        clean_pred.append(
            (
                a,
                b
            )
        )


        previous_offset = b


    if clean_pred:

        clips_with_events += 1


    total_events += len(
        clean_pred
    )


    recs.append({

        "clip_id":
            r["clip_id"],

        "events": [

            {
                "onset":
                    round(
                        float(a),
                        3
                    ),

                "offset":
                    round(
                        float(b),
                        3
                    )
            }

            for a, b in clean_pred
        ]
    })


# ============================================================
# WRITE JSONL
# ============================================================

OUT_JSONL = os.path.join(
    OUT_DIR,
    "predictions_tuned.jsonl"
)


with open(
    OUT_JSONL,
    "w",
    encoding="utf-8"
) as f:

    for r in recs:

        f.write(
            json.dumps(
                r,
                ensure_ascii=False
            )
            + "\n"
        )


# ============================================================
# ZIP
# ============================================================

OUT_ZIP = os.path.join(
    OUT_DIR,
    "predictions_tuned.zip"
)


with zipfile.ZipFile(
    OUT_ZIP,
    "w",
    zipfile.ZIP_DEFLATED
) as z:

    z.write(
        OUT_JSONL,
        "predictions.jsonl"
    )


# ============================================================
# FINAL SANITY CHECK
# ============================================================

assert len(recs) == len(
    records
)


print("\n==============================")
print("TUNED PREDICTIONS READY")
print("==============================")

print(
    "clips:",
    len(recs)
)

print(
    "clips with >=1 event:",
    clips_with_events
)

print(
    "total predicted events:",
    total_events
)

print(
    "best parameters:",
    BEST_PARAMS
)

print(
    "JSONL:",
    OUT_JSONL
)

print(
    "ZIP:",
    OUT_ZIP
)


print("\nFirst 3 predictions:")

for r in recs[:3]:
    print(r)

Validation: /content/drive/MyDrive/PHD/Datathon/validation
Checkpoint: /content/drive/MyDrive/Datathon/work/ckpt/boundary_wavlm_beats_verified_stable_best.pt

Loaded best checkpoint
epoch: 2
internal val loss: 0.40138038700983464

Validation clips: 2501
Natural WAVs : 1806
Synthetic WAVs: 695

Running model over official validation...
1/2501
250/2501
500/2501
750/2501
1000/2501
1250/2501
1500/2501
1750/2501
2000/2501
2250/2501
2500/2501
missing audio: 0
Cached: /content/drive/MyDrive/Datathon/work/official_val_posteriors_epoch_2.pkl

Inference records: 2501

POSTERIOR DISTRIBUTION
p000: 0.0027
p010: 0.0070
p025: 0.0139
p050: 0.0725
p075: 0.5644
p090: 0.9138
p095: 0.9607
p099: 0.9836
p100: 0.9945

BEST RAW THRESHOLDS
thr=0.225 frameDice=0.7291
thr=0.250 frameDice=0.7287
thr=0.200 frameDice=0.7287
thr=0.275 frameDice=0.7281
thr=0.175 frameDice=0.7272
thr=0.300 frameDice=0.7271
thr=0.325 frameDice=0.7256
thr=0.150 frameDice=0.7248
thr=0.350 frameDice=0.7235
thr=0.375 frameDice=0.7208

Rou

In [ ]:
# ============================================================
# FINAL COMPETITION SUBMISSION
# WavLM + BEATs + tuned boundary decoder
#
# Tuned on official validation:
#   threshold = 0.325
#   median    = 0 ms
#   min_dur   = 0.060 sec
#   merge_gap = 0 sec
#   snapping  = True
#
# Produces:
#   predictions.jsonl
#   submission_WavLM_BEATs_tuned.zip
# ============================================================

import os
import glob
import json
import time
import zipfile
import numpy as np
import soundfile as sf
import librosa
import torch


# ============================================================
# SETTINGS
# ============================================================

BEST_PATH = (
    "/content/drive/MyDrive/Datathon/work/ckpt/"
    "boundary_wavlm_beats_verified_stable_best.pt"
)

WORK = "/content/drive/MyDrive/Datathon/work"

TEST_ROOT = "/content/drive/MyDrive/PHD/Datathon/"

INPUT_ZIP = (
    "/content/drive/MyDrive/Datathon/input_data.zip"
)

# Tuned decoder parameters
THR = 0.325
MED_MS = 0
MIN_DUR = 0.060
MERGE_GAP = 0.0

USE_SNAP = True
SNAP_WINDOW = 0.20


# ============================================================
# LOAD BEST CHECKPOINT
# ============================================================

assert os.path.isfile(
    BEST_PATH
), f"Checkpoint missing: {BEST_PATH}"

print("Loading checkpoint:")
print(BEST_PATH)

try:
    ckpt = torch.load(
        BEST_PATH,
        map_location="cpu",
        weights_only=False
    )
except TypeError:
    ckpt = torch.load(
        BEST_PATH,
        map_location="cpu"
    )


missing, unexpected = model.load_state_dict(
    ckpt["model"],
    strict=False
)

# Frozen WavLM/BEATs weights are intentionally not stored
bad_missing = [
    k for k in missing
    if not (
        k.startswith("wavlm.")
        or k.startswith("beats.")
    )
]

if bad_missing:
    print(
        "WARNING unexpected missing keys:",
        bad_missing[:10]
    )

if unexpected:
    print(
        "WARNING unexpected keys:",
        unexpected[:10]
    )


model = model.to(device)
model.eval()

print(
    "Best epoch:",
    ckpt.get("epoch", "?")
)

print(
    "Best internal val loss:",
    ckpt.get("best_val_loss", "?")
)


# ============================================================
# FIND / EXTRACT TEST AUDIO
# ============================================================

wavs_test = sorted(
    glob.glob(
        os.path.join(
            TEST_ROOT,

            "*.wav"
        ),
        recursive=True
    )
)


if not wavs_test:

    if not os.path.isfile(INPUT_ZIP):

        raise FileNotFoundError(
            f"No WAV files under {TEST_ROOT} "
            f"and input ZIP missing: {INPUT_ZIP}"
        )

    print(
        "\nExtracting test data..."
    )

    os.makedirs(
        TEST_ROOT,
        exist_ok=True
    )

    with zipfile.ZipFile(
        INPUT_ZIP,
        "r"
    ) as z:

        z.extractall(
            TEST_ROOT
        )


    wavs_test = sorted(
        glob.glob(
            os.path.join(
                TEST_ROOT,

                "*.wav"
            ),
            recursive=True
        )
    )


print(
    "\nTest WAV files:",
    len(wavs_test)
)

if len(wavs_test) == 0:
    raise RuntimeError(
        "No test WAV files found."
    )


# Expected from your previous test package
if len(wavs_test) != 5517:

    print(
        "WARNING: expected ~5517 test clips "
        f"but found {len(wavs_test)}."
    )


# ============================================================
# CHECK DUPLICATE CLIP IDs
# ============================================================

def clip_id_from_path(path):

    return os.path.splitext(
        os.path.basename(path)
    )[0]


clip_ids = [
    clip_id_from_path(p)
    for p in wavs_test
]


if len(set(clip_ids)) != len(clip_ids):

    from collections import Counter

    c = Counter(clip_ids)

    dup = [
        k for k, v in c.items()
        if v > 1
    ]

    raise RuntimeError(
        f"Duplicate clip IDs found: {dup[:10]}"
    )


print(
    "Unique clip IDs:",
    len(set(clip_ids))
)


# ============================================================
# DECODER
# ============================================================

def decode_tuned(
    prob,
    duration,
    thr=THR,
    min_dur=MIN_DUR,
    merge_gap=MERGE_GAP
):

    # No median filtering:
    # validation tuning selected MED_MS = 0
    mask = (
        prob >= thr
    )

    intervals = []

    start = None


    # ----------------------------------------
    # binary mask -> intervals
    # ----------------------------------------
    for i, active in enumerate(mask):

        if active and start is None:

            start = i

        elif (
            not active
            and start is not None
        ):

            intervals.append(
                [
                    start * FRAME,
                    i * FRAME
                ]
            )

            start = None


    if start is not None:

        intervals.append(
            [
                start * FRAME,
                len(mask) * FRAME
            ]
        )


    # ----------------------------------------
    # merge gaps
    # ----------------------------------------
    merged = []


    for a, b in intervals:

        if (
            merged
            and
            a - merged[-1][1]
            <= merge_gap
        ):

            merged[-1][1] = b

        else:

            merged.append(
                [a, b]
            )


    # ----------------------------------------
    # duration filtering
    # ----------------------------------------
    output = []


    for a, b in merged:

        a = max(
            0.0,
            float(a)
        )

        b = min(
            float(duration),
            float(b)
        )


        if b - a >= min_dur:

            output.append(
                (
                    a,
                    b
                )
            )


    return output


# ============================================================
# BOUNDARY SNAPPING
# ============================================================

def snap_tuned(
    candidates,
    onset_dist,
    offset_dist,
    duration,
    window=SNAP_WINDOW,
    min_dur=MIN_DUR
):

    if not candidates:
        return []


    N = len(
        onset_dist
    )


    W = max(
        1,
        int(
            round(
                window / FRAME
            )
        )
    )


    output = []


    for onset, offset in candidates:

        on_frame = int(
            round(
                onset / FRAME
            )
        )

        off_frame = int(
            round(
                offset / FRAME
            )
        )


        # ------------------------------------
        # onset
        # ------------------------------------
        a = max(
            0,
            on_frame - W
        )

        b = min(
            N,
            on_frame + W + 1
        )


        best_on = onset


        if b > a:

            best_on = (
                a
                + int(
                    np.argmin(
                        onset_dist[a:b]
                    )
                )
            ) * FRAME


        # ------------------------------------
        # offset
        # ------------------------------------
        a = max(
            0,
            off_frame - W
        )

        b = min(
            N,
            off_frame + W + 1
        )


        best_off = offset


        if b > a:

            best_off = (
                a
                + int(
                    np.argmin(
                        offset_dist[a:b]
                    )
                )
            ) * FRAME


        best_on = max(
            0.0,
            float(best_on)
        )

        best_off = min(
            float(duration),
            float(best_off)
        )


        if (
            best_off
            - best_on
            >= min_dur
        ):

            output.append(
                (
                    best_on,
                    best_off
                )
            )


    return output


# ============================================================
# TEST INFERENCE
# ============================================================

predictions = {}

posterior_stats = []

t0 = time.time()

print("\nStarting test inference...\n")


model.eval()


with torch.inference_mode():

    for i, path in enumerate(
        wavs_test,
        start=1
    ):

        cid = clip_id_from_path(
            path
        )


        # ------------------------------------
        # audio
        # ------------------------------------
        y, sr0 = sf.read(
            path,
            dtype="float32",
            always_2d=False
        )


        y = np.asarray(
            y,
            dtype=np.float32
        )


        if y.ndim == 2:

            y = y.mean(
                axis=1
            )


        if sr0 != SR:

            y = librosa.resample(
                y,
                orig_sr=sr0,
                target_sr=SR
            ).astype(
                np.float32
            )


        y = np.nan_to_num(
            y,
            nan=0.0,
            posinf=0.0,
            neginf=0.0
        ).astype(
            np.float32
        )


        if len(y) == 0:

            predictions[cid] = []

            print(
                "WARNING empty audio:",
                cid
            )

            continue


        valid_samples = len(y)

        duration = (
            valid_samples
            / float(SR)
        )


        # ------------------------------------
        # SAME mel frontend as training
        # ------------------------------------
        mel = logmel(y)


        mel = (
            mel - mel.mean()
        ) / (
            mel.std() + 1e-5
        )


        mel = np.asarray(
            mel,
            dtype=np.float32
        )


        T = mel.shape[1]


        need = (
            T * HOP
        )


        yy = np.pad(
            y,
            (
                0,
                max(
                    0,
                    need - valid_samples
                )
            )
        )[:need].astype(
            np.float32
        )


        # ------------------------------------
        # tensors
        # ------------------------------------
        xt = (
            torch
            .from_numpy(mel)
            [None, None]
            .to(
                device,
                dtype=torch.float32
            )
        )


        wt = (
            torch
            .from_numpy(yy)
            [None]
            .to(
                device,
                dtype=torch.float32
            )
        )


        wl = torch.tensor(
            [
                min(
                    valid_samples,
                    need
                )
            ],
            dtype=torch.long,
            device=device
        )


        # ------------------------------------
        # model
        # ------------------------------------
        p_noise, p_on, p_off = model(
            xt,
            wt,
            wl
        )


        if not (
            torch.isfinite(p_noise).all()
            and torch.isfinite(p_on).all()
            and torch.isfinite(p_off).all()
        ):

            raise FloatingPointError(
                f"Non-finite output for {cid}"
            )


        prob = (
            torch.sigmoid(
                p_noise[0, :T]
            )
            .float()
            .cpu()
            .numpy()
        )


        onset_dist = (
            p_on[0, :T]
            .float()
            .cpu()
            .numpy()
        )


        offset_dist = (
            p_off[0, :T]
            .float()
            .cpu()
            .numpy()
        )


        posterior_stats.append(
            (
                float(
                    prob.mean()
                ),
                float(
                    prob.max()
                )
            )
        )


        # ------------------------------------
        # tuned decode
        # ------------------------------------
        candidates = decode_tuned(
            prob,
            duration
        )


        if USE_SNAP:

            events = snap_tuned(
                candidates,
                onset_dist,
                offset_dist,
                duration
            )

        else:

            events = candidates


        # ------------------------------------
        # sort / sanitize
        # ------------------------------------
        events = sorted(
            events,
            key=lambda x: x[0]
        )


        clean_events = []

        previous_offset = -1.0


        for onset, offset in events:

            onset = max(
                0.0,
                float(onset)
            )

            offset = min(
                float(duration),
                float(offset)
            )


            if onset < previous_offset:

                onset = previous_offset


            if offset <= onset:
                continue


            if (
                offset - onset
                < MIN_DUR
            ):
                continue


            clean_events.append(
                (
                    onset,
                    offset
                )
            )


            previous_offset = offset


        predictions[cid] = (
            clean_events
        )


        # ------------------------------------
        # progress
        # ------------------------------------
        if (
            i == 1
            or i % 250 == 0
            or i == len(wavs_test)
        ):

            total_so_far = sum(
                len(x)
                for x in predictions.values()
            )

            with_events = sum(
                1
                for x in predictions.values()
                if len(x) > 0
            )

            elapsed = (
                time.time() - t0
            ) / 60


            print(
                f"{i}/{len(wavs_test)} "
                f"| events={total_so_far} "
                f"| clips+={with_events} "
                f"| {elapsed:.1f} min",
                flush=True
            )


# ============================================================
# GLOBAL DIAGNOSTICS
# ============================================================

total_events = sum(
    len(v)
    for v in predictions.values()
)

clips_with_events = sum(
    bool(v)
    for v in predictions.values()
)


print("\n")
print("=" * 60)
print("TEST INFERENCE SUMMARY")
print("=" * 60)

print(
    "clips:",
    len(predictions)
)

print(
    "clips with >=1 event:",
    clips_with_events
)

print(
    "clips with no event:",
    len(predictions)
    - clips_with_events
)

print(
    "total predicted events:",
    total_events
)

print(
    "events / clip:",
    total_events
    / max(
        len(predictions),
        1
    )
)


if posterior_stats:

    means = np.array(
        [
            x[0]
            for x in posterior_stats
        ]
    )

    maxima = np.array(
        [
            x[1]
            for x in posterior_stats
        ]
    )


    print("\nPosterior statistics:")

    print(
        "mean clip posterior:",
        means.mean()
    )

    print(
        "median clip maximum:",
        np.median(maxima)
    )

    print(
        "p90 clip maximum:",
        np.percentile(
            maxima,
            90
        )
    )

    print(
        "maximum:",
        maxima.max()
    )


# ============================================================
# IMPORTANT SAFETY CHECK
# ============================================================

if total_events == 0:

    raise RuntimeError(
        "\nZERO EVENTS PRODUCED.\n"
        "DO NOT SUBMIT THIS FILE."
    )


# If almost nothing is predicted, flag it.
if clips_with_events < (
    0.10
    * len(predictions)
):

    raise RuntimeError(
        f"Only {clips_with_events}/"
        f"{len(predictions)} clips contain events. "
        "Submission looks suspicious; inspect before uploading."
    )


# ============================================================
# CREATE JSONL
# ============================================================

submission_rows = []


for cid in sorted(
    predictions.keys()
):

    events = predictions[cid]


    row = {

        "clip_id":
            cid,

        "events": [

            {
                "onset":
                    round(
                        float(onset),
                        3
                    ),

                "offset":
                    round(
                        float(offset),
                        3
                    )
            }

            for onset, offset
            in events
        ]
    }


    submission_rows.append(
        row
    )


# ============================================================
# VALIDATE STRUCTURE
# ============================================================

assert (
    len(submission_rows)
    ==
    len(wavs_test)
)


ids = [
    r["clip_id"]
    for r in submission_rows
]


assert (
    len(ids)
    ==
    len(set(ids))
)


for row in submission_rows:

    previous_offset = -1.0


    for event in row["events"]:

        onset = event["onset"]
        offset = event["offset"]


        assert isinstance(
            onset,
            (int, float)
        )

        assert isinstance(
            offset,
            (int, float)
        )


        assert (
            0
            <= onset
            < offset
        )


        assert (
            onset
            >= previous_offset
            - 1e-6
        )


        previous_offset = (
            offset
        )


# ============================================================
# WRITE predictions.jsonl
# ============================================================

JSONL_PATH = os.path.join(
    WORK,
    "predictions.jsonl"
)


with open(
    JSONL_PATH,
    "w",
    encoding="utf-8"
) as f:

    for row in submission_rows:

        f.write(
            json.dumps(
                row,
                ensure_ascii=False
            )
            + "\n"
        )


# ============================================================
# ZIP
# ============================================================

ZIP_PATH = os.path.join(
    WORK,
    "submission_WavLM_BEATs_tuned.zip"
)


with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    zipfile.ZIP_DEFLATED
) as z:

    # IMPORTANT:
    # filename inside ZIP must be predictions.jsonl

    z.write(
        JSONL_PATH,
        arcname="predictions.jsonl"
    )


# ============================================================
# VERIFY ZIP
# ============================================================

with zipfile.ZipFile(
    ZIP_PATH,
    "r"
) as z:

    names = z.namelist()


assert names == [
    "predictions.jsonl"
]


# ============================================================
# RE-READ FILE AND VERIFY COUNTS
# ============================================================

check_rows = []


with open(
    JSONL_PATH,
    "r",
    encoding="utf-8"
) as f:

    for line in f:

        check_rows.append(
            json.loads(line)
        )


assert (
    len(check_rows)
    ==
    len(wavs_test)
)


check_events = sum(
    len(
        r["events"]
    )
    for r in check_rows
)


assert (
    check_events
    ==
    total_events
)


# ============================================================
# FINAL OUTPUT
# ============================================================

print("\n")
print("=" * 60)
print("FINAL SUBMISSION READY")
print("=" * 60)

print(
    "checkpoint epoch:",
    ckpt.get(
        "epoch",
        "?"
    )
)

print(
    "threshold:",
    THR
)

print(
    "median filter:",
    MED_MS,
    "ms"
)

print(
    "minimum duration:",
    MIN_DUR
)

print(
    "merge gap:",
    MERGE_GAP
)

print(
    "boundary snapping:",
    USE_SNAP
)

print(
    "\nclips:",
    len(check_rows)
)

print(
    "clips with events:",
    clips_with_events
)

print(
    "total events:",
    check_events
)


print(
    "\nJSONL:"
)

print(
    JSONL_PATH
)


print(
    "\nSUBMIT THIS ZIP:"
)

print(
    ZIP_PATH
)


print(
    "\nFirst 5 rows:"
)

for r in check_rows[:5]:
    print(r)

Loading checkpoint:
/content/drive/MyDrive/Datathon/work/ckpt/boundary_wavlm_beats_verified_stable_best.pt
Best epoch: 2
Best internal val loss: 0.40138038700983464

Test WAV files: 5517
Unique clip IDs: 5517

Starting test inference...

1/5517 | events=1 | clips+=1 | 0.0 min
250/5517 | events=404 | clips+=247 | 0.6 min
500/5517 | events=803 | clips+=497 | 1.0 min
750/5517 | events=1248 | clips+=747 | 1.5 min
1000/5517 | events=1611 | clips+=994 | 2.0 min
1250/5517 | events=1952 | clips+=1243 | 2.5 min
1500/5517 | events=2310 | clips+=1490 | 3.0 min
1750/5517 | events=2663 | clips+=1736 | 3.5 min
2000/5517 | events=3109 | clips+=1984 | 4.1 min
2250/5517 | events=3587 | clips+=2234 | 4.6 min
2500/5517 | events=4014 | clips+=2483 | 5.2 min
2750/5517 | events=4428 | clips+=2733 | 5.7 min
3000/5517 | events=4847 | clips+=2981 | 6.2 min
3250/5517 | events=5218 | clips+=3229 | 6.7 min
3500/5517 | events=5644 | clips+=3479 | 7.2 min
3750/5517 | events=6036 | clips+=3727 | 7.7 min
4000/5517 | 